# Evidence-Grounded Paper Q&A v2 — Colab Enterprise (L4) 초고속 vLLM 원샷 워크벤치 (v5.2 Battle-Tested Final)

- **실제 Colab Enterprise L4(Python 3.13 / `transformers 5.x` / 22.03 GiB VRAM / `/content` NVMe) 실측 검증 반영 완료**:
  - **100% `/content` 디스크 전용**: `HF_HOME`, `TMPDIR`, `PIP_CACHE_DIR`, `UV_CACHE_DIR`, `/content/vllm_venv`, `HF_HUB_DISABLE_XET=1` 및 `/root/.cache/huggingface` 심볼릭 링크로 루트(`/`) 디스크 풀 **0%**.
  - **격리 Python 3.12 `vLLM` 서버 + Guided JSON(`json_object`)**: `Qwen3-4B-Instruct-2507 (BF16)`(`~2.0s/q`) + `Qwen3-14B-AWQ (Marlin)`(`~3.7s/q`) 고속 추론 + `json_fallback` **0%** + 학습 전 OS 레벨 프로세스 종료로 좀비 VRAM **0%**.
  - **arXiv 차단 회피 & 안정화**: `ccdv/arxiv-summarization` HF Parquet 스트리밍(150편 10초), `mDeBERTa-v3` **FP32 명시 + 자동 NaN 건전성 검증**, `TrainingArguments`/`SFTConfig`/`DPOConfig` 시그니처 자동 필터링(`transformers 5.x` / `trl` 완벽 호환).
  - **스마트 고속 이어하기(Fast-Resume)**: 이미 끝난 셀의 체크포인트(`120/120`, `150편`, `600청크`)가 있으면 vLLM 재기동 없이 0.1초 만에 건너뛰고 남은 구간부터 바로 이어 달립니다.

## Section 0. 최종 원샷 마스터 제어판 · GPU/저장소 초기화 · 격리 vLLM(Python 3.12) & 패키지 설치

In [ ]:
# Cell 0.1 — 최종 원샷 제어판 + 저장소/메모리/vLLM 프로세스 안전 유틸리티
import gc
import os
import shutil
import signal
import subprocess
import sys
import zipfile
from pathlib import Path

!nvidia-smi

# ============================================================================
# [최종 v5.0 원샷 마스터 설정] — 그대로 "모두 실행(Run All)" 하시면 약 2.5~3.5시간 완주
# ============================================================================
ONE_SHOT_FULL_RUN = True          # True: Phase 0~4 + Holdout 회귀 + GGUF/ONNX 내보내기 원샷 실행
USE_VLLM_SERVER = True            # True: 격리 Python 3.12 vLLM 서버(10~15배 고속), 실패 시 PyTorch 자동 폴백
VLLM_GPU_MEM_UTIL = 0.68          # L4(22.03 GiB): vLLM ~15.0 GiB + 인코더/FP32 NLI 공존 여유 ~7.0 GiB 확보!
VLLM_MAX_MODEL_LEN = 6144         # 5-Chunk Parent(1800) + 1024 생성 토큰 완벽 수용
USE_GDRIVE_MOUNT = False          # False: 중간 OAuth 팝업 방지 (끝나고 Cell 6.4에서 한 번에 Drive 복사 가능)
GCS_BUCKET = ""                   # 선택: "gs://your-bucket/PaperQnA_v2" (설정 시 단계별 자동 백업)
HF_TOKEN = ""                     # 선택: Hugging Face 속도 제한 방지 토큰

# 데이터·동시성 규모 (L4 24GB + vLLM 최적 튜닝값)
EXT_TRAIN_PAPERS = 150            # 외부 arXiv Full-PDF 학습 논문 수 (corpus30 0% 겹침)
MIX_QASPER_SAMPLES = 500          # 실제 인간 작성 과학 논문 QA(QASPER) 혼합 수
TEACHER_THINKING_SYNTH = False    # False 권장: JSON 절단 0% + 2.5~3배 고속 + 150편 다양성 확보
HOLDOUT_V2_PAPERS = 30            # 신규 미공개 평가셋 동결 논문 수

BATCH_SIZE_4B = 16                # vLLM 동시 요청 수 (PyTorch 폴백 시 자동으로 4 이하로 제한)
BATCH_SIZE_14B = 12               # vLLM 14B-AWQ Marlin 동시 요청 수 (PyTorch 폴백 시 4 이하)
DPO_ROLLOUT_BATCH = 6             # vLLM On-Policy 병렬 롤아웃 (6 x 4 = 24 병렬, Prefix Caching 적용!)

# 단계별 개별 스위치 (필요 시 특정 구간만 False로 끌 수 있음)
RUN_QWEN3_RETRIEVAL_AB = ONE_SHOT_FULL_RUN
RUN_TEACHER_BASELINE = ONE_SHOT_FULL_RUN
RUN_SEMANTIC_JUDGE = ONE_SHOT_FULL_RUN
RUN_NLI_SYNTH = ONE_SHOT_FULL_RUN
RUN_NLI_TRAIN = ONE_SHOT_FULL_RUN
RUN_EXP2_ABLATION = ONE_SHOT_FULL_RUN
RUN_QA_SYNTH = ONE_SHOT_FULL_RUN
RUN_SFT_TRAIN = ONE_SHOT_FULL_RUN
RUN_ONPOLICY_DPO_COLLECT = ONE_SHOT_FULL_RUN
RUN_DPO_TRAIN = ONE_SHOT_FULL_RUN
RUN_FINAL_HOLDOUT_REGRESSION = ONE_SHOT_FULL_RUN
RUN_EXPORT_GGUF = ONE_SHOT_FULL_RUN
RUN_EXPORT_NLI_ONNX = ONE_SHOT_FULL_RUN
RUN_BUILD_HOLDOUT_V2 = ONE_SHOT_FULL_RUN

# ============================================================================
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"   # Xet 이중 캐시로 인한 디스크 풀 원천 차단!

# Colab Enterprise의 대용량 NVMe(/content)로 HF/임시 캐시 전면 배치 + /root/.cache/huggingface 심볼릭 링크!
for _nvme_dir in (Path("/content/hf_cache"), Path("/content/tmp"), Path("/content/xdg_cache")):
    _nvme_dir.mkdir(parents=True, exist_ok=True)
_root_hf = Path("/root/.cache/huggingface")
_root_hf.parent.mkdir(parents=True, exist_ok=True)
if _root_hf.exists() and not _root_hf.is_symlink():
    for _it in list(_root_hf.iterdir()):
        _dst = Path("/content/hf_cache") / _it.name
        if not _dst.exists():
            shutil.move(str(_it), str(_dst))
    shutil.rmtree(_root_hf, ignore_errors=True)
if not _root_hf.is_symlink():
    _root_hf.symlink_to(Path("/content/hf_cache"))
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/content/hf_cache/hub"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TMPDIR"] = "/content/tmp"
os.environ["XDG_CACHE_HOME"] = "/content/xdg_cache"
os.environ["UV_CACHE_DIR"] = "/content/tmp/uv_cache"
os.environ["UV_PYTHON_INSTALL_DIR"] = "/content/tmp/uv_python"
os.environ["PIP_CACHE_DIR"] = "/content/tmp/pip_cache"
os.environ["TORCH_HOME"] = "/content/xdg_cache/torch"
os.environ["TRITON_CACHE_DIR"] = "/content/tmp/triton_cache"
os.environ["VLLM_CACHE_ROOT"] = "/content/tmp/vllm_cache"
!df -h / /content

BUNDLE_NAME = "paper_qna_colab_bundle.zip"
DRIVE_FOLDER_NAME = "PaperQnA_v2"

if USE_GDRIVE_MOUNT:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        WORK_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
        STORAGE_MODE = "gdrive"
    except Exception as exc:
        print(f"[INFO] Drive 마운트 불가 -> 로컬 경로 폴백 ({exc})")
        WORK_DIR = Path("/content") / DRIVE_FOLDER_NAME
        STORAGE_MODE = "gcs" if GCS_BUCKET else "local"
else:
    WORK_DIR = Path("/content") / DRIVE_FOLDER_NAME
    STORAGE_MODE = "gcs" if GCS_BUCKET else "local"

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] WORK_DIR={WORK_DIR} (mode={STORAGE_MODE}, ONE_SHOT={ONE_SHOT_FULL_RUN}, papers={EXT_TRAIN_PAPERS})")


def sync_from_gcs(rel_path: str):
    if not GCS_BUCKET:
        return
    dst = WORK_DIR / rel_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["gcloud", "storage", "cp", "-r", f"{GCS_BUCKET}/{rel_path}", str(dst.parent)], check=False)


def sync_to_gcs(rel_path: str):
    if not GCS_BUCKET:
        return
    src = WORK_DIR / rel_path
    if src.exists():
        subprocess.run(["gcloud", "storage", "cp", "-r", str(src), f"{GCS_BUCKET}/{rel_path}"], check=False)


if GCS_BUCKET and not (WORK_DIR / BUNDLE_NAME).exists():
    sync_from_gcs(BUNDLE_NAME)

BUNDLE_DIR = Path("/content/paper_qna_bundle")
RUNS_DIR = WORK_DIR / "runs"
DATASETS_DIR = WORK_DIR / "datasets"
MODELS_DIR = WORK_DIR / "models"
EXPORTS_DIR = WORK_DIR / "exports"
HOLDOUT_V2_DIR = WORK_DIR / "corpus_v2_holdout"
for d in (BUNDLE_DIR, RUNS_DIR, DATASETS_DIR, MODELS_DIR, EXPORTS_DIR, HOLDOUT_V2_DIR):
    d.mkdir(parents=True, exist_ok=True)

bundle_candidates = [
    Path("/content") / BUNDLE_NAME,
    WORK_DIR / BUNDLE_NAME,
    Path.cwd() / BUNDLE_NAME,
    Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME / BUNDLE_NAME,
]
BUNDLE_ZIP = next((p for p in bundle_candidates if p.is_file()), None)
if BUNDLE_ZIP is None:
    raise FileNotFoundError(
        f"{BUNDLE_NAME}을 찾을 수 없습니다. Colab 왼쪽 파일 패널(/content/)에 업로드한 뒤 다시 실행하세요."
    )

with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
    zf.extractall(BUNDLE_DIR)

if str(BUNDLE_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR / "scripts"))

print(f"[OK] 번들 해제 완료: {BUNDLE_ZIP} -> {BUNDLE_DIR}")


def free_vram(*var_names):
    '''전역 변수·IPython 예외 트레이스백에 묶인 좀비 GPU 텐서까지 완전히 회수합니다.'''
    import traceback, torch
    for attr in ("last_traceback", "last_value", "last_type", "last_exc"):
        val = getattr(sys, attr, None)
        if attr == "last_traceback" and val is not None:
            traceback.clear_frames(val)
        if attr == "last_exc" and val is not None and getattr(val, "__traceback__", None) is not None:
            traceback.clear_frames(val.__traceback__)
        if hasattr(sys, attr):
            try:
                setattr(sys, attr, None)
            except Exception:
                pass
    g = globals()
    for item in var_names:
        if isinstance(item, str):
            g.pop(item, None)
        else:
            for k in [k for k, v in list(g.items()) if v is item]:
                g.pop(k, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"[VRAM] allocated={alloc:.2f} GiB / {total:.2f} GiB")


def set_encoders_device(target_device: str):
    '''학습(SFT/DPO/NLI) 전 인코더를 CPU로 내리고, 추론 시 GPU 복원 (인코더 자체 VRAM만 합산 + OOM 시 CPU 안전 폴백!).'''
    import torch
    g = globals()
    mods = []
    if "embedder" in g and g["embedder"] is not None:
        mods.append(("embedder", g["embedder"], False))
    if "reranker" in g and g["reranker"] is not None:
        mods.append(("reranker", g["reranker"].model, False))
    for nli_key in ("nli_scorer", "active_nli", "final_nli"):
        if nli_key in g and g[nli_key] is not None:
            mods.append((nli_key, g[nli_key].model, True))

    actual_device = target_device
    if target_device == "cuda" and torch.cuda.is_available():
        free_b, _ = torch.cuda.mem_get_info()
        enc_cuda_b = 0
        for _, m, _ in mods:
            try:
                for t in list(m.parameters()) + list(m.buffers()):
                    if isinstance(t, torch.Tensor) and t.is_cuda:
                        enc_cuda_b += t.numel() * t.element_size()
            except Exception:
                pass
        avail_gb = (free_b + enc_cuda_b) / (1024 ** 3)
        if avail_gb < 2.8:
            print(f"[ENCODERS] GPU 인코더 가용 {avail_gb:.2f} GiB < 2.8 GiB -> CPU(fp32) 유지", flush=True)
            actual_device = "cpu"

    def _move_all(dev: str):
        dt = torch.float16 if dev == "cuda" else torch.float32
        for k, m, force_fp32 in mods:
            m.to(device=dev, dtype=torch.float32 if force_fp32 else dt)
            if force_fp32:
                g[k].device = dev

    try:
        _move_all(actual_device)
    except Exception as _enc_exc:
        print(f"[ENCODERS] GPU 이동 폴백({type(_enc_exc).__name__}) -> CPU(fp32) 유지", flush=True)
        actual_device = "cpu"
        torch.cuda.empty_cache()
        _move_all("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        free_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
        print(f"[ENCODERS -> {actual_device}] PyTorch={alloc:.2f} GiB | GPU Free={free_gb:.2f} GiB", flush=True)


def cleanup_disk_caches(drop_hf_names: tuple[str, ...] = ()):
    '''uv/pip/xet 임시 캐시 및 불필요한 HF 모델 캐시를 비워 디스크 여유를 확보합니다.'''
    for p in (Path("/root/.cache/uv"), Path("/root/.cache/pip"), Path("/root/.cache/huggingface/xet"), Path("/content/tmp/uv_cache"), Path("/content/hf_cache/xet")):
        shutil.rmtree(p, ignore_errors=True)
    for hub_dir in (Path("/content/hf_cache/hub"), Path("/root/.cache/huggingface/hub")):
        if hub_dir.exists() and drop_hf_names:
            for d in list(hub_dir.glob("models--*")):
                if any(d.name == name or d.name.endswith(f"--{name}") for name in drop_hf_names):
                    shutil.rmtree(d, ignore_errors=True)
    u_root = shutil.disk_usage("/")
    u_cnt = shutil.disk_usage("/content") if Path("/content").exists() else u_root
    print(f"[DISK] / Free={u_root.free / (1024**3):.1f} GiB | /content Free={u_cnt.free / (1024**3):.1f} GiB", flush=True)


def cleanup_all_gpu_llms():
    if "stop_vllm_server" in globals():
        stop_vllm_server()
    g = globals()
    for _k in ("nli_scorer", "active_nli", "final_nli"):
        if _k in g and g[_k] is not None:
            try:
                g[_k].model.cpu()
            except Exception:
                pass
    free_vram(
        "llm_4b", "tok_4b", "llm_14b", "tok_14b",
        "sft_llm", "sft_tok", "dpo_llm", "dpo_tok",
        "final_llm", "final_tok", "nli_scorer", "active_nli", "final_nli",
        "model", "base", "trainer", "nli_mod", "q3_emb", "q3_rmod", "q3_rtok",
    )

In [ ]:
# Cell 0.2 — 필수 라이브러리 설치 + 격리 Python 3.12 vLLM 가상환경(`/content/vllm_venv`) 원클릭 구축 + API 사전 검증
%pip uninstall -y -q torchao
%pip install -q --upgrade-strategy only-if-needed "transformers>=4.51.0" "sentence-transformers>=3.3.0" "rank-bm25>=0.2.2" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" "peft>=0.14.0" "trl>=0.15.0" "datasets>=3.0.0" "scikit-learn>=1.5.0" "scipy>=1.13.0" "pandas>=2.2.0" "pydantic>=2.7.0" "pypdf>=5.0.0" "pymupdf4llm>=0.0.17" "uv>=0.6.0"

VLLM_VENV_DIR = Path("/content/vllm_venv")
VLLM_BIN = VLLM_VENV_DIR / "bin" / "vllm"

if USE_VLLM_SERVER and not VLLM_BIN.exists():
    print("[SETUP] 격리된 Python 3.12 환경(/content/vllm_venv)에 vLLM 설치 중 (약 1~2분)...", flush=True)
    r_venv = subprocess.run(["uv", "venv", str(VLLM_VENV_DIR), "--python", "3.12", "--seed", "--managed-python"], check=False)
    if r_venv.returncode == 0:
        r_inst = subprocess.run(
            ["uv", "pip", "install", "--python", str(VLLM_VENV_DIR / "bin" / "python"), "vllm>=0.8.5", "--torch-backend=auto"],
            check=False,
        )
        if r_inst.returncode != 0:
            subprocess.run(["uv", "pip", "install", "--python", str(VLLM_VENV_DIR / "bin" / "python"), "vllm"], check=False)
    cleanup_disk_caches()

VLLM_READY = bool(USE_VLLM_SERVER and VLLM_BIN.exists())
import inspect, torch, transformers, sentence_transformers, peft, trl
from transformers import TrainingArguments
from trl import SFTConfig, DPOConfig
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__} | CUDA={torch.cuda.is_available()} | GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | TF32=ON")
print(f"Transformers : {transformers.__version__} | SentTrans={sentence_transformers.__version__}")
print(f"PEFT / TRL   : {peft.__version__} / {trl.__version__}")
print(f"vLLM Isolated: {'READY (' + str(VLLM_BIN) + ')' if VLLM_READY else 'FALLBACK -> PyTorch SDPA'}")

# 1초 Preflight 자가 점검 (TrainingArguments / SFTConfig / DPOConfig 호환성 사전 확인!)
for _cls, _name in [(TrainingArguments, "TrainingArguments"), (SFTConfig, "SFTConfig"), (DPOConfig, "DPOConfig")]:
    _sig = inspect.signature(_cls.__init__).parameters
    _test_kw = {"output_dir": "/content/tmp/preflight", "warmup_steps": 10, "bf16": False, "report_to": "none"}
    _ = _cls(**{k: v for k, v in _test_kw.items() if k in _sig})
print("[PREFLIGHT OK] TrainingArguments / SFTConfig / DPOConfig 초기화 검증 통과!", flush=True)

## Section 1. `corpus30` 기준선 · PDF 텍스트 보정 · 통계 검정(McNemar & Paper-Cluster Bootstrap)

In [ ]:
# Cell 1.1 — corpus30 로드 + 결정론적 PDF 공백/합자 보정 + 누수 차단 가드(30편)
import hashlib
import inspect
import json
import random
import re
import statistics
import time
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.stats import binomtest

import bm25_retrieval
from grounded_answer_contract import (
    ABSTENTION_ANSWERS,
    AnswerValidationError,
    canonical_abstention_payload,
    validate_answer_payload,
)


def read_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def repair_pdf_chunk_text(text: str) -> str:
    t = unicodedata.normalize("NFKC", text).replace("\u00ad", "").replace("\u200b", "")
    t = re.sub(r"\b(this|that|these|those|when|where|with|from|into|such|each)(?=[a-z]{6,}\b)", r"\1 ", t)
    t = re.sub(r"(?<=\bWhen)([A-Z])(?=(?:and|or)\b)", r" \1 ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


CHUNKS = read_jsonl(BUNDLE_DIR / "data/processed/chunks.jsonl")
for _c in CHUNKS:
    _c["raw_text"] = _c["text"]
    _c["text"] = repair_pdf_chunk_text(_c["text"])

PAGES = read_jsonl(BUNDLE_DIR / "data/processed/pages.jsonl")
QUESTIONS = read_jsonl(BUNDLE_DIR / "data/benchmarks/corpus30/questions.jsonl")
MANIFEST = json.loads((BUNDLE_DIR / "data/benchmarks/corpus30/manifest.json").read_text(encoding="utf-8"))
V1_DEV_REPORT = json.loads((BUNDLE_DIR / "data/benchmarks/corpus30/results/final-dev-report.json").read_text(encoding="utf-8"))
V1_HOLDOUT_REPORT = json.loads((BUNDLE_DIR / "data/benchmarks/corpus30/results/final-holdout-report.json").read_text(encoding="utf-8"))

CHUNKS_BY_PAPER = defaultdict(list)
for ch in CHUNKS:
    CHUNKS_BY_PAPER[ch["paper_id"]].append(ch)
for pid in CHUNKS_BY_PAPER:
    CHUNKS_BY_PAPER[pid].sort(key=lambda c: (c["page"], c["chunk_id"]))

DEV_QUESTIONS = [q for q in QUESTIONS if q["split"] == "dev"]
HOLDOUT_QUESTIONS = [q for q in QUESTIONS if q["split"] == "holdout"]
QUESTION_MAP = {q["case_id"]: q for q in QUESTIONS}

FORBIDDEN_ARXIV_IDS = set()
FORBIDDEN_TITLES = set()
for p in MANIFEST["papers"]:
    FORBIDDEN_TITLES.add(re.sub(r"\s+", " ", p["title"].casefold()).strip())
    m = re.search(r"arxiv\.org/abs/([0-9.]+)", p.get("source_url", ""))
    if m:
        FORBIDDEN_ARXIV_IDS.add(m.group(1))

print(f"Papers={len(CHUNKS_BY_PAPER)} | Chunks={len(CHUNKS)} | DEV={len(DEV_QUESTIONS)} | HOLDOUT={len(HOLDOUT_QUESTIONS)}")
print(f"Leakage Guard locked: {len(FORBIDDEN_ARXIV_IDS)} arXiv IDs / {len(FORBIDDEN_TITLES)} titles")

display(pd.DataFrame([
    {"scope": "v1_final_DEV_paired", **V1_DEV_REPORT["metrics"]},
    {"scope": "v1_final_DEV_en", **V1_DEV_REPORT["by_language"]["en"]},
    {"scope": "v1_final_DEV_ko", **V1_DEV_REPORT["by_language"]["ko"]},
])[["scope", "contract_valid", "answerable_retained", "all_fact_aliases", "acceptable_source_page_hit", "unsupported_refused", "false_premise_refused", "mean_wall_seconds"]])

In [ ]:
# Cell 1.2 — corpus_benchmark.py 호환 평가기 + McNemar 검정 & Paper-Cluster Bootstrap 95% CI
EXPERIMENT_LOG = RUNS_DIR / "experiment_log.jsonl"


def _norm_eval(text: str) -> str:
    text = re.sub(r"(?<=\d)\s*\.\s*(?=\d)", ".", text)
    return re.sub(r"\s+", " ", text.casefold().replace("–", "-").replace("−", "-"))


def _contains_alias(text: str, alias: str) -> bool:
    t, a = _norm_eval(text), _norm_eval(alias)
    if re.fullmatch(r"[\d,.]+", a):
        return bool(re.search(r"(?<![\d.])" + re.escape(a) + r"(?![\d.])", t))
    return a in t


def evaluate_predictions(records: list[dict], split: str = "dev", questions: list[dict] | None = None) -> dict:
    pool = questions if questions is not None else [q for q in QUESTIONS if split == "all" or q["split"] == split]
    lookup = {q["case_id"]: q for q in pool}
    evaluated = []

    for rec in records:
        cid = rec["case_id"]
        if cid not in lookup:
            continue
        case = lookup[cid]
        out = rec.get("output") or {}
        resp = out.get("response") or {}
        cited = out.get("cited_evidence") or []
        retrieved = out.get("retrieved_evidence") or []

        contract = True
        try:
            validate_answer_payload(resp, cited, case["language"])
        except (ValueError, TypeError, KeyError):
            contract = False

        answered = contract and resp.get("sufficiency") == "sufficient"
        refused = contract and resp.get("sufficiency") == "insufficient" and not cited
        facts = [
            any(_contains_alias(resp.get("answer", ""), alias) for alias in fact["any_of"])
            for fact in case.get("required_facts", [])
        ]
        gold_pages = set(case.get("gold_pages", []))
        ret_pages = {e["page"] for e in retrieved}
        cit_pages = {e["page"] for e in cited}

        en_case = lookup.get(re.sub(r"-ko$", "-en", cid), case)
        ret_text = " ".join(e.get("text", "") for e in retrieved)
        ret_facts = [
            any(_contains_alias(ret_text, alias) for alias in fact["any_of"])
            for fact in en_case.get("required_facts", [])
        ]
        pipe = out.get("pipeline") or {}
        evaluated.append({
            "case_id": cid, "meaning_id": case["meaning_id"], "paper_id": case["paper_id"],
            "split": case["split"], "language": case["language"], "answerability": case["answerability"],
            "contract_valid": contract, "answered": bool(answered), "refused": bool(refused),
            "fact_alias_hits": sum(facts), "required_fact_count": len(facts),
            "all_fact_aliases": bool(answered and facts and all(facts)),
            "acceptable_source_page_hit": bool(answered and (gold_pages & cit_pages)),
            "ret_gold_page_hit": bool(gold_pages & ret_pages),
            "ret_all_en_facts_hit": bool(ret_facts and all(ret_facts)),
            "quote_verified": bool(pipe.get("quote_verified", True)),
            "numeric_verified": bool(pipe.get("numeric_verified", True)),
            "wall_seconds": rec.get("wall_seconds", 0.0),
        })

    def _summarize(rows: list[dict], paired: bool) -> dict:
        grouped = {}
        for r in rows:
            grouped.setdefault(r["meaning_id"] if paired else r["case_id"], []).append(r)
        pos = [g for g in grouped.values() if g[0]["answerability"] == "answerable"]
        neg = [g for g in grouped.values() if g[0]["answerability"] != "answerable"]
        ans_rows = [r for r in rows if r["answered"]]
        return {
            "n_outputs": len(rows),
            "contract_valid": sum(r["contract_valid"] for r in rows),
            "ret_gold_page_hit": sum(all(r["ret_gold_page_hit"] for r in g) for g in pos),
            "ret_all_en_facts": sum(all(r["ret_all_en_facts_hit"] for r in g) for g in pos),
            "answerable_retained": sum(all(r["answered"] for r in g) for g in pos),
            "all_fact_aliases": sum(all(r["all_fact_aliases"] for r in g) for g in pos),
            "acceptable_source_page_hit": sum(all(r["acceptable_source_page_hit"] for r in g) for g in pos),
            "unsupported_refused": sum(all(r["refused"] for r in g) for g in neg if g[0]["answerability"] == "unsupported"),
            "false_premise_refused": sum(all(r["refused"] for r in g) for g in neg if g[0]["answerability"] == "false_premise"),
            "negative_answers": sum(any(r["answered"] for r in g) for g in neg),
            "quote_verified_rate": round(sum(r["quote_verified"] for r in ans_rows) / max(len(ans_rows), 1), 3),
            "numeric_verified_rate": round(sum(r["numeric_verified"] for r in ans_rows) / max(len(ans_rows), 1), 3),
            "mean_wall_seconds": round(statistics.mean(r["wall_seconds"] for r in rows), 3) if rows else 0.0,
        }

    return {
        "split": split,
        "paired": _summarize(evaluated, paired=True),
        "en": _summarize([r for r in evaluated if r["language"] == "en"], paired=False),
        "ko": _summarize([r for r in evaluated if r["language"] == "ko"], paired=False),
        "records": evaluated,
    }


def compare_experiments_statistically(base_report: dict, cand_report: dict, n_boot: int = 1000, seed: int = 378) -> pd.DataFrame:
    b_map = {r["case_id"]: r for r in base_report["records"]}
    c_map = {r["case_id"]: r for r in cand_report["records"]}
    common = sorted(set(b_map) & set(c_map))
    rng = np.random.default_rng(seed)

    specs = [
        ("answerable_retained (all)", "answerable", "answered"),
        ("all_fact_aliases (all)", "answerable", "all_fact_aliases"),
        ("acceptable_source_page_hit", "answerable", "acceptable_source_page_hit"),
        ("unsupported_refused", "unsupported", "refused"),
        ("false_premise_refused", "false_premise", "refused"),
    ]
    rows = []
    for label, atype, field in specs:
        sub = [cid for cid in common if b_map[cid]["answerability"] == atype]
        if not sub:
            continue
        b_vals = np.array([int(b_map[cid][field]) for cid in sub])
        c_vals = np.array([int(c_map[cid][field]) for cid in sub])
        n01 = int(((b_vals == 0) & (c_vals == 1)).sum())
        n10 = int(((b_vals == 1) & (c_vals == 0)).sum())
        p_val = float(binomtest(n01, n01 + n10, 0.5).pvalue) if (n01 + n10) > 0 else 1.0

        by_p = defaultdict(list)
        for idx, cid in enumerate(sub):
            by_p[b_map[cid]["paper_id"]].append(idx)
        pids = list(by_p.keys())
        deltas = []
        for _ in range(n_boot):
            sampled_pids = rng.choice(pids, size=len(pids), replace=True)
            idxs = [i for p in sampled_pids for i in by_p[p]]
            deltas.append(float(c_vals[idxs].mean() - b_vals[idxs].mean()))
        lo, hi = np.percentile(deltas, [2.5, 97.5])
        rows.append({
            "metric": label, "n_outputs": len(sub),
            "base": f"{b_vals.mean():.3f}", "cand": f"{c_vals.mean():.3f}",
            "delta": f"{(c_vals.mean() - b_vals.mean()):+.3f}",
            "95%_cluster_CI": f"[{lo:+.3f}, {hi:+.3f}]",
            "gain/loss": f"+{n01}/-{n10}", "mcnemar_p": round(p_val, 4),
        })
    df = pd.DataFrame(rows)
    display(df)
    return df


def log_experiment(exp_name: str, report: dict, notes: str = "", semantic_summary: dict | None = None) -> pd.DataFrame:
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "exp_name": exp_name, "split": report["split"], "notes": notes,
        "paired": report["paired"], "en": report["en"], "ko": report["ko"],
        "semantic": semantic_summary,
    }
    existing = read_jsonl(EXPERIMENT_LOG) if EXPERIMENT_LOG.exists() else []
    already_same = any(e.get("exp_name") == exp_name and e.get("split") == report["split"] and e.get("paired") == report["paired"] for e in existing)
    if not already_same:
        with EXPERIMENT_LOG.open("a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        sync_to_gcs("runs/experiment_log.jsonl")
        print(f"[LOGGED] {exp_name} -> {EXPERIMENT_LOG}")
    else:
        print(f"[LOG EXISTS] {exp_name} ({report['split']}) 이미 기록됨 (중복 생략)")
    return show_experiment_table()


def show_experiment_table() -> pd.DataFrame:
    rows = [
        {
            "exp": "v1_final_DEV (baseline)",
            "contract": f"{V1_DEV_REPORT['metrics']['contract_valid']}/120",
            "pair_ans": f"{V1_DEV_REPORT['metrics']['answerable_retained']}/40",
            "ko_ans": f"{V1_DEV_REPORT['by_language']['ko']['answerable_retained']}/40",
            "pair_facts": f"{V1_DEV_REPORT['metrics']['all_fact_aliases']}/40",
            "en/ko_facts": f"{V1_DEV_REPORT['by_language']['en']['all_fact_aliases']}/{V1_DEV_REPORT['by_language']['ko']['all_fact_aliases']}",
            "unsup_ref": f"{V1_DEV_REPORT['metrics']['unsupported_refused']}/10",
            "fp_en/ko/pair": f"{V1_DEV_REPORT['by_language']['en']['false_premise_refused']}/{V1_DEV_REPORT['by_language']['ko']['false_premise_refused']}/{V1_DEV_REPORT['metrics']['false_premise_refused']}",
            "quote/num": "extractive", "sem_comp": "-",
            "sec/q": round(V1_DEV_REPORT["metrics"]["mean_wall_seconds"], 2),
        },
        {
            "exp": "v1_final_HOLDOUT (baseline)",
            "contract": f"{V1_HOLDOUT_REPORT['metrics']['contract_valid']}/240",
            "pair_ans": f"{V1_HOLDOUT_REPORT['metrics']['answerable_retained']}/80",
            "ko_ans": f"{V1_HOLDOUT_REPORT['by_language']['ko']['answerable_retained']}/80",
            "pair_facts": f"{V1_HOLDOUT_REPORT['metrics']['all_fact_aliases']}/80",
            "en/ko_facts": f"{V1_HOLDOUT_REPORT['by_language']['en']['all_fact_aliases']}/{V1_HOLDOUT_REPORT['by_language']['ko']['all_fact_aliases']}",
            "unsup_ref": f"{V1_HOLDOUT_REPORT['metrics']['unsupported_refused']}/20",
            "fp_en/ko/pair": f"{V1_HOLDOUT_REPORT['by_language']['en']['false_premise_refused']}/{V1_HOLDOUT_REPORT['by_language']['ko']['false_premise_refused']}/{V1_HOLDOUT_REPORT['metrics']['false_premise_refused']}",
            "quote/num": "extractive", "sem_comp": "-",
            "sec/q": round(V1_HOLDOUT_REPORT["metrics"]["mean_wall_seconds"], 2),
        },
    ]
    if EXPERIMENT_LOG.exists():
        for e in read_jsonl(EXPERIMENT_LOG):
            p, en, ko = e["paired"], e["en"], e["ko"]
            da = 40 if e["split"] == "dev" else 80
            dn = 10 if e["split"] == "dev" else 20
            sem = e.get("semantic") or {}
            rows.append({
                "exp": f"{e['exp_name']} ({e['split']})",
                "contract": f"{p['contract_valid']}/{p['n_outputs']}",
                "pair_ans": f"{p['answerable_retained']}/{da}",
                "ko_ans": f"{ko['answerable_retained']}/{da}",
                "pair_facts": f"{p['all_fact_aliases']}/{da}",
                "en/ko_facts": f"{en['all_fact_aliases']}/{ko['all_fact_aliases']}",
                "unsup_ref": f"{p['unsupported_refused']}/{dn}",
                "fp_en/ko/pair": f"{en['false_premise_refused']}/{ko['false_premise_refused']}/{p['false_premise_refused']}",
                "quote/num": f"{p.get('quote_verified_rate', 0):.2f}/{p.get('numeric_verified_rate', 0):.2f}",
                "sem_comp": f"{sem.get('paired_completeness', '-'):.2f}" if isinstance(sem.get("paired_completeness"), (int, float)) else "-",
                "sec/q": round(p["mean_wall_seconds"], 2),
            })
    df = pd.DataFrame(rows)
    display(df)
    return df


print("[OK] 평가기 · McNemar 검정 · Paper-Cluster Bootstrap 95% CI 준비 완료")

## Section 2. [Phase 1B] 검색 · 재정렬 업그레이드 (`BGE-M3` vs `Qwen3-0.6B` A/B + Dual-Score Rerank)

In [ ]:
# Cell 2.1 — Stack A (`BGE-M3` + `bge-reranker-v2-m3`) + Dual-Score Rerank + Parent(1800자)
import torch
import torch.nn.functional as F
from sentence_transformers import CrossEncoder, SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_MODEL_NAME = "BAAI/bge-m3"
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
PARENT_MAX_CHARS = 1800
EMBED_CACHE = RUNS_DIR / "bge_m3_corpus30_v4.npz"

embedder = SentenceTransformer(
    EMBED_MODEL_NAME, device=DEVICE,
    model_kwargs={"torch_dtype": torch.float16} if DEVICE == "cuda" else {},
)
reranker = CrossEncoder(
    RERANK_MODEL_NAME, max_length=1024, device=DEVICE,
    automodel_args={"torch_dtype": torch.float16} if DEVICE == "cuda" else {},
)

chunk_ids_all = [c["chunk_id"] for c in CHUNKS]
ALL_EMBEDDINGS = None
if EMBED_CACHE.exists():
    z = np.load(EMBED_CACHE, allow_pickle=False)
    if list(z["chunk_ids"]) == chunk_ids_all:
        ALL_EMBEDDINGS = z["embeddings"].astype(np.float32)
        print(f"[CACHE HIT] {EMBED_CACHE} ({ALL_EMBEDDINGS.shape})")

if ALL_EMBEDDINGS is None:
    t0 = time.perf_counter()
    ALL_EMBEDDINGS = embedder.encode(
        [c["text"] for c in CHUNKS], batch_size=32, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)
    np.savez_compressed(EMBED_CACHE, chunk_ids=np.array(chunk_ids_all), embeddings=ALL_EMBEDDINGS)
    print(f"[SAVED] {EMBED_CACHE} in {time.perf_counter()-t0:.1f}s")

CHUNK_IDX_BY_ID = {c["chunk_id"]: i for i, c in enumerate(CHUNKS)}
BM25_BY_PAPER = {pid: bm25_retrieval.build_index(chs) for pid, chs in CHUNKS_BY_PAPER.items()}
EMB_BY_PAPER = {
    pid: np.stack([ALL_EMBEDDINGS[CHUNK_IDX_BY_ID[c["chunk_id"]]] for c in chs], axis=0)
    for pid, chs in CHUNKS_BY_PAPER.items()
}


def _rrf_fuse(rank_lists: list[list[dict]], rrf_k: int = 60, top_k: int = 25) -> list[dict]:
    scores, by_id = defaultdict(float), {}
    for rlist in rank_lists:
        for rank, item in enumerate(rlist, start=1):
            cid = item["chunk"]["chunk_id"]
            scores[cid] += 1.0 / (rrf_k + rank)
            by_id[cid] = item["chunk"]
    ranked = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))[:top_k]
    return [{"rank": i, "score": float(sc), "chunk": by_id[cid]} for i, (cid, sc) in enumerate(ranked, start=1)]


def _expand_parent_window(selected_chunks: list[dict], paper_chunks: list[dict], max_chars: int = PARENT_MAX_CHARS) -> list[dict]:
    pos_map = {c["chunk_id"]: idx for idx, c in enumerate(paper_chunks)}
    expanded = []
    for rank, item in enumerate(selected_chunks, start=1):
        ch = item["chunk"]
        idx = pos_map[ch["chunk_id"]]
        parts = [ch["text"]]
        if idx - 1 >= 0 and paper_chunks[idx - 1]["page"] == ch["page"] and len(paper_chunks[idx - 1]["text"]) + len(parts[0]) <= max_chars:
            parts.insert(0, paper_chunks[idx - 1]["text"])
        if idx + 1 < len(paper_chunks) and paper_chunks[idx + 1]["page"] == ch["page"] and sum(len(p) for p in parts) + len(paper_chunks[idx + 1]["text"]) <= max_chars:
            parts.append(paper_chunks[idx + 1]["text"])
        expanded.append({
            "evidence_id": f"ev-{rank:02d}-{ch['chunk_id']}",
            "paper_id": ch["paper_id"], "chunk_id": ch["chunk_id"],
            "page": ch["page"], "section": ch.get("section") or "Body",
            "text": " ".join(parts), "core_text": ch["text"],
            "reranker_score": item["reranker_score"], "retrieval_score": item["retrieval_score"],
        })
    return expanded


def retrieve_from_chunks(
    query: str,
    paper_chunks: list[dict],
    paper_embs: np.ndarray,
    bm25_idx,
    en_query: str | None = None,
    candidate_k: int = 25,
    top_k: int = 5,
    max_per_page: int = 2,
    use_parent_window: bool = True,
    en_rerank_weight: float = 0.65,
) -> dict:
    queries_for_dense = [query] + ([en_query] if en_query and en_query != query else [])
    q_embs = embedder.encode(queries_for_dense, normalize_embeddings=True).astype(np.float32)

    rank_lists = []
    for q_text, q_emb in zip(queries_for_dense, q_embs):
        rank_lists.append(bm25_retrieval.rank_chunks(bm25_idx, paper_chunks, q_text, top_k=20))
        sims = (paper_embs @ q_emb).tolist()
        dense_ranked = sorted(zip(sims, paper_chunks), key=lambda x: (-x[0], x[1]["chunk_id"]))[:20]
        rank_lists.append([{"rank": r, "score": float(s), "chunk": c} for r, (s, c) in enumerate(dense_ranked, start=1)])

    fused = _rrf_fuse(rank_lists, rrf_k=60, top_k=candidate_k)
    if en_query and en_query != query:
        s_en = reranker.predict([(en_query, it["chunk"]["text"]) for it in fused], batch_size=32, show_progress_bar=False)
        s_ko = reranker.predict([(query, it["chunk"]["text"]) for it in fused], batch_size=32, show_progress_bar=False)
        scores = (en_rerank_weight * np.asarray(s_en) + (1.0 - en_rerank_weight) * np.asarray(s_ko)).tolist()
    else:
        scores = reranker.predict([(query, it["chunk"]["text"]) for it in fused], batch_size=32, show_progress_bar=False).tolist()

    scored = [
        {"chunk": it["chunk"], "reranker_score": float(sc), "retrieval_score": float(it["score"])}
        for sc, it in zip(scores, fused)
    ]
    scored.sort(key=lambda x: (-x["reranker_score"], x["chunk"]["chunk_id"]))

    selected, page_counts = [], Counter()
    for item in scored:
        pg = item["chunk"]["page"]
        if page_counts[pg] >= max_per_page:
            continue
        selected.append(item)
        page_counts[pg] += 1
        if len(selected) >= top_k:
            break

    evidence = (
        _expand_parent_window(selected, paper_chunks, max_chars=PARENT_MAX_CHARS)
        if use_parent_window
        else [
            {
                "evidence_id": f"ev-{r:02d}-{it['chunk']['chunk_id']}",
                "paper_id": it["chunk"]["paper_id"], "chunk_id": it["chunk"]["chunk_id"],
                "page": it["chunk"]["page"], "section": it["chunk"].get("section") or "Body",
                "text": it["chunk"]["text"], "core_text": it["chunk"]["text"],
                "reranker_score": it["reranker_score"], "retrieval_score": it["retrieval_score"],
            }
            for r, it in enumerate(selected, start=1)
        ]
    )
    return {
        "evidence": evidence,
        "top_reranker_score": scored[0]["reranker_score"] if scored else -99.0,
        "score_margin_1_5": (scored[0]["reranker_score"] - scored[min(4, len(scored) - 1)]["reranker_score"]) if scored else 0.0,
    }


def retrieve_v2(query: str, paper_id: str, **kwargs) -> dict:
    return retrieve_from_chunks(
        query, CHUNKS_BY_PAPER[paper_id], EMB_BY_PAPER[paper_id], BM25_BY_PAPER[paper_id], **kwargs
    )


print(f"[OK] Stack A 검색기 준비 완료 (Parent max_chars={PARENT_MAX_CHARS})")

In [ ]:
# Cell 2.2 — [Exp 1B] DEV 검색·재정렬 A/B 비교 (Stack A: BGE-M3 vs Stack B: Qwen3-0.6B)
ret_stats = defaultdict(list)
t0 = time.perf_counter()
for q in DEV_QUESTIONS:
    res = retrieve_v2(q["question"], q["paper_id"], en_query=None, use_parent_window=False)
    lang, atype = q["language"], q["answerability"]
    ret_stats[(lang, atype, "top_score")].append(res["top_reranker_score"])
    if atype == "answerable":
        gold_pages = set(q["gold_pages"])
        ret_pages = {e["page"] for e in res["evidence"]}
        ret_stats[(lang, "gold_page_hit")].append(bool(gold_pages & ret_pages))
        en_q = QUESTION_MAP[q["case_id"].replace("-ko", "-en")]
        ret_text = " ".join(e["text"] for e in res["evidence"])
        all_f = all(any(_contains_alias(ret_text, a) for a in f["any_of"]) for f in en_q["required_facts"])
        ret_stats[(lang, "all_facts_in_top5")].append(all_f)

print(f"[Stack A: BGE-M3 + bge-reranker-v2-m3] DEV 120문항 ({time.perf_counter()-t0:.1f}s)")
for lang in ("en", "ko"):
    gp = sum(ret_stats[(lang, "gold_page_hit")])
    af = sum(ret_stats[(lang, "all_facts_in_top5")])
    med_a = statistics.median(ret_stats[(lang, "answerable", "top_score")])
    med_u = statistics.median(ret_stats[(lang, "unsupported", "top_score")])
    print(f"  [{lang.upper()}] gold_page_top5={gp}/40 ({100*gp/40:.1f}%) | all_facts_top5={af}/40 ({100*af/40:.1f}%) | med(ans={med_a:.2f}, unsup={med_u:.2f})")

if RUN_QWEN3_RETRIEVAL_AB and (RUNS_DIR / "exp1b_retrieval_ab.json").exists():
    print(f"[CACHE HIT] exp1b_retrieval_ab.json -> {(RUNS_DIR / 'exp1b_retrieval_ab.json').read_text(encoding='utf-8').strip()}")
elif RUN_QWEN3_RETRIEVAL_AB:
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        print("\n[Stack B: Qwen3-Embedding-0.6B + Qwen3-Reranker-0.6B A/B 측정 중...]")
        q3_emb = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"torch_dtype": torch.float16})
        q3_rtok = AutoTokenizer.from_pretrained("Qwen/Qwen3-Reranker-0.6B", padding_side="left")
        q3_rmod = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-Reranker-0.6B", torch_dtype=torch.float16).to(DEVICE).eval()
        t_no, t_yes = q3_rtok.convert_tokens_to_ids("no"), q3_rtok.convert_tokens_to_ids("yes")
        r_prefix = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
        r_suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

        @torch.no_grad()
        def _q3_rerank(query_text: str, docs: list[str], batch_size: int = 10) -> list[float]:
            instr = "Given a scientific paper question, retrieve relevant passages from the paper that answer it"
            prompts = [f"{r_prefix}<Instruct>: {instr}\n<Query>: {query_text}\n<Document>: {d[:1800]}{r_suffix}" for d in docs]
            out_scores = []
            for i in range(0, len(prompts), batch_size):
                enc = q3_rtok(prompts[i:i + batch_size], padding=True, truncation=True, max_length=2048, return_tensors="pt").to(DEVICE)
                logits = q3_rmod(**enc).logits[:, -1, :]
                probs = F.log_softmax(torch.stack([logits[:, t_no], logits[:, t_yes]], dim=1), dim=1)[:, 1].exp().tolist()
                out_scores.extend(probs)
            return out_scores

        q3_all_embs = q3_emb.encode([c["text"] for c in CHUNKS], batch_size=32, normalize_embeddings=True).astype(np.float32)
        q3_by_p = {pid: np.stack([q3_all_embs[CHUNK_IDX_BY_ID[c["chunk_id"]]] for c in chs], axis=0) for pid, chs in CHUNKS_BY_PAPER.items()}
        ab_hits = Counter()
        for q in DEV_QUESTIONS:
            if q["answerability"] != "answerable":
                continue
            pchs = CHUNKS_BY_PAPER[q["paper_id"]]
            qe = q3_emb.encode([q["question"]], prompt_name="query", normalize_embeddings=True).astype(np.float32)[0]
            sims = (q3_by_p[q["paper_id"]] @ qe).tolist()
            d_top = [{"rank": r, "score": float(s), "chunk": c} for r, (s, c) in enumerate(sorted(zip(sims, pchs), key=lambda x: (-x[0], x[1]["chunk_id"]))[:20], 1)]
            b_top = bm25_retrieval.rank_chunks(BM25_BY_PAPER[q["paper_id"]], pchs, q["question"], top_k=20)
            fused = _rrf_fuse([b_top, d_top], top_k=20)
            r_scores = _q3_rerank(q["question"], [f["chunk"]["text"] for f in fused])
            top5 = [f["chunk"] for _, f in sorted(zip(r_scores, fused), key=lambda x: -x[0])[:5]]
            if set(q["gold_pages"]) & {c["page"] for c in top5}:
                ab_hits[q["language"]] += 1
        ab_summary = {"stack_a_en": sum(ret_stats[("en", "gold_page_hit")]), "stack_a_ko": sum(ret_stats[("ko", "gold_page_hit")]), "stack_b_en": ab_hits["en"], "stack_b_ko": ab_hits["ko"]}
        (RUNS_DIR / "exp1b_retrieval_ab.json").write_text(json.dumps(ab_summary, indent=2) + "\n", encoding="utf-8")
        print(f"  [Stack B Qwen3-0.6B] EN={ab_hits['en']}/40 | KO={ab_hits['ko']}/40 (saved exp1b_retrieval_ab.json)")
        free_vram("q3_emb", "q3_rmod", "q3_rtok")
        cleanup_disk_caches(("Qwen3-Embedding-0.6B", "Qwen3-Reranker-0.6B"))
    except Exception as exc:
        print(f"[WARN] Stack B A/B 건너뜀 (Stack A로 계속 진행): {exc}")

## Section 3. [Phase 1C] 초고속 격리 vLLM 추론 엔진 (`Continuous Batching` + `Prefix Caching`) · Base 4B & Teacher 14B · Semantic Judge

- **10~15배 고속화**: `/opt/vllm_venv` 격리 서버에서 `Qwen3-4B-Instruct-2507 (BF16)`과 `Qwen3-14B-AWQ (Marlin)`를 서빙하고, 학습(`NLI`/`SFT`/`DPO`) 직전에는 프로세스 그룹을 완전히 종료해 VRAM `0.00 GiB`를 확보합니다.

In [ ]:
# Cell 3.1 — 초고속 격리 vLLM 서버 매니저 + PyTorch 자동 폴백 · Quote+수치 결정론 검증기
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_STUDENT_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
TEACHER_MODEL_NAME = "Qwen/Qwen3-14B"
TEACHER_VLLM_MODEL = "Qwen/Qwen3-14B-AWQ"
TEACHER_VLLM_FP8_MODEL = "Qwen/Qwen3-14B-FP8"
VLLM_PORT = 8000
VLLM_LOG_PATH = Path("/content/vllm_server.log")
VLLM_TMP_MERGED = Path("/content/tmp_vllm_merged")
NUM_PATTERN = re.compile(r"(?<![A-Za-z0-9])[-+−]?(?:\d+(?:[,.]\d+)*|\.\d+)(?:[eE][-+−]?\d+)?")

_ACTIVE_VLLM_PROC = None
_LOCAL_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


@dataclass
class VLLMHandle:
    base_url: str
    served_model: str
    source_model: str
    port: int


def stop_vllm_server(wait_vram: bool = True, clean_merged: bool = True):
    global _ACTIVE_VLLM_PROC
    if _ACTIVE_VLLM_PROC is not None:
        try:
            pgid = os.getpgid(_ACTIVE_VLLM_PROC.pid)
            os.killpg(pgid, signal.SIGTERM)
            try:
                _ACTIVE_VLLM_PROC.wait(timeout=8)
            except subprocess.TimeoutExpired:
                os.killpg(pgid, signal.SIGKILL)
                _ACTIVE_VLLM_PROC.wait(timeout=5)
        except Exception:
            pass
        _ACTIVE_VLLM_PROC = None
    subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["pkill", "-9", "-f", "vllm_venv/bin/vllm"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if clean_merged:
        shutil.rmtree(VLLM_TMP_MERGED, ignore_errors=True)
    if wait_vram and torch.cuda.is_available():
        for _ in range(20):
            free_gb, total_gb = [b / (1024 ** 3) for b in torch.cuda.mem_get_info()]
            if free_gb >= total_gb - 3.5:
                break
            time.sleep(0.5)
        free_gb, total_gb = [b / (1024 ** 3) for b in torch.cuda.mem_get_info()]
        print(f"[vLLM STOPPED] GPU Free={free_gb:.2f} / {total_gb:.2f} GiB", flush=True)


def _extract_vllm_root_error(log_path: Path) -> str:
    if not log_path.exists():
        return "log missing"
    lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    hits = [i for i, ln in enumerate(lines) if any(k in ln for k in ("Error:", "Exception:", "OutOfMemory", "ValueError", "RuntimeError", "NotImplementedError", "ModuleNotFoundError"))]
    if hits:
        idx = hits[0]
        return "\n".join(lines[max(0, idx - 6):min(len(lines), idx + 12)])
    return "\n".join(lines[-35:])


def _start_vllm_server(
    model_target: str,
    served_name: str,
    quantization: str | None = None,
    port: int = VLLM_PORT,
    gpu_util: float | None = None,
    kv_cache_dtype: str | None = None,
    enforce_eager: bool = True,
) -> VLLMHandle:
    global _ACTIVE_VLLM_PROC
    keep_tmp = str(model_target).startswith(str(VLLM_TMP_MERGED))
    stop_vllm_server(wait_vram=True, clean_merged=not keep_tmp)
    set_encoders_device("cpu")
    free_vram()

    util = gpu_util if gpu_util is not None else VLLM_GPU_MEM_UTIL
    if torch.cuda.is_available():
        _f_gb, _t_gb = [b / (1024 ** 3) for b in torch.cuda.mem_get_info()]
        max_safe_util = max(0.45, round((_f_gb - 1.2) / _t_gb, 2))
        if util > max_safe_util:
            print(f"[vLLM UTIL ADJUST] free={_f_gb:.2f} GiB -> util {util} -> {max_safe_util}", flush=True)
            util = max_safe_util
    is_awq = "awq" in str(model_target).lower()
    is_fp8 = "fp8" in str(model_target).lower()
    cmd = [
        str(VLLM_BIN), "serve", str(model_target),
        "--served-model-name", served_name,
        "--host", "127.0.0.1", "--port", str(port),
        "--max-model-len", str(VLLM_MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(util),
        "--enable-prefix-caching",
        "--disable-log-requests",
    ]
    if is_awq:
        cmd += ["--dtype", "float16"]
    elif not is_fp8:
        cmd += ["--dtype", "bfloat16"]
    if quantization:
        cmd += ["--quantization", quantization]
    if kv_cache_dtype:
        cmd += ["--kv-cache-dtype", kv_cache_dtype]
    if enforce_eager:
        cmd += ["--enforce-eager"]

    env = os.environ.copy()
    env["VLLM_USAGE_STATS_DISABLED"] = "1"
    env["HF_HUB_DISABLE_XET"] = "1"
    cleanup_disk_caches()
    t0 = time.perf_counter()
    print(f"[vLLM START] model={model_target} | quant={quantization or 'auto'} | util={util} | eager={enforce_eager} ...", flush=True)
    log_f = VLLM_LOG_PATH.open("w", encoding="utf-8")
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    _ACTIVE_VLLM_PROC = proc
    health_url = f"http://127.0.0.1:{port}/health"
    ready = False
    for _ in range(360):
        if proc.poll() is not None:
            break
        try:
            with _LOCAL_OPENER.open(health_url, timeout=2) as r:
                if r.status == 200:
                    ready = True
                    break
        except Exception:
            time.sleep(1.0)
    if ready:
        free_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3) if torch.cuda.is_available() else 0.0
        print(f"[vLLM READY] {model_target} in {time.perf_counter()-t0:.1f}s | GPU Free={free_gb:.2f} GiB", flush=True)
        try:
            set_encoders_device(DEVICE)
        except Exception as _se:
            print(f"[WARN] 인코더 GPU 복원 생략 (vLLM은 정상 유지): {_se}", flush=True)
        return VLLMHandle(base_url=f"http://127.0.0.1:{port}", served_model=served_name, source_model=str(model_target), port=port)
    root_err = _extract_vllm_root_error(VLLM_LOG_PATH)
    stop_vllm_server(wait_vram=True)
    raise RuntimeError(f"vLLM 기동 실패 ({model_target}):\n{root_err}")


def load_hf_llm(model_name: str, load_in_4bit: bool = False, lora_adapter: str | Path | list | None = None):
    adapters = [lora_adapter] if isinstance(lora_adapter, (str, Path)) else (list(lora_adapter) if lora_adapter else [])
    adapters = [Path(a) for a in adapters if a and Path(a).exists()]

    if globals().get("VLLM_READY", False) and USE_VLLM_SERVER:
        try:
            if adapters:
                stop_vllm_server(wait_vram=True)
                set_encoders_device("cpu")
                print(f"[vLLM MERGE] CPU에서 LoRA 순차 병합 중: {[a.name for a in adapters]} -> {VLLM_TMP_MERGED}", flush=True)
                from peft import PeftModel
                merged = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="cpu")
                tok = AutoTokenizer.from_pretrained(model_name)
                for ad in adapters:
                    merged = PeftModel.from_pretrained(merged, str(ad)).merge_and_unload()
                shutil.rmtree(VLLM_TMP_MERGED, ignore_errors=True)
                merged.save_pretrained(str(VLLM_TMP_MERGED), safe_serialization=True)
                tok.save_pretrained(str(VLLM_TMP_MERGED))
                del merged, tok
                gc.collect()
                handle = _start_vllm_server(str(VLLM_TMP_MERGED), served_name="active_llm", quantization=None, enforce_eager=True)
                return handle, None
            if load_in_4bit and model_name == TEACHER_MODEL_NAME:
                cleanup_disk_caches(("Qwen3-Embedding-0.6B", "Qwen3-Reranker-0.6B"))
                teacher_candidates = [
                    (TEACHER_VLLM_MODEL, "awq_marlin", 0.72, None, True),
                    (TEACHER_VLLM_FP8_MODEL, None, 0.82, "fp8", True),
                    (TEACHER_VLLM_MODEL, "awq", 0.72, None, True),
                ]
                errs = []
                for m_id, q_flg, g_util, kv_dt, e_flag in teacher_candidates:
                    try:
                        return _start_vllm_server(m_id, served_name="active_llm", quantization=q_flg, gpu_util=g_util, kv_cache_dtype=kv_dt, enforce_eager=e_flag), None
                    except Exception as te:
                        errs.append(f"[{m_id}/{q_flg}] {str(te)[:220]}")
                        print(f"[vLLM RETRY] {m_id}({q_flg}) 실패 -> 다음 후보 시도...", flush=True)
                raise RuntimeError(" | ".join(errs))
            handle = _start_vllm_server(model_name, served_name="active_llm", quantization=None, enforce_eager=True)
            return handle, None
        except Exception as exc:
            print(f"[WARN] vLLM 기동 실패 -> PyTorch SDPA 자동 폴백: {exc}", flush=True)
            stop_vllm_server(wait_vram=True)

    print(f"Loading HF PyTorch LLM: {model_name} (4bit={load_in_4bit}, adapters={[a.name for a in adapters]}) ...", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs = {"attn_implementation": "sdpa"}
    if load_in_4bit:
        kwargs["device_map"] = "auto"
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.bfloat16
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    if not load_in_4bit and torch.cuda.is_available():
        model = model.to("cuda")
    if adapters:
        from peft import PeftModel
        for ad in adapters:
            model = PeftModel.from_pretrained(model, str(ad))
            if not load_in_4bit:
                model = model.merge_and_unload()
    model.eval()
    return model, tokenizer


def _render_prompt(tokenizer, messages: list[dict], enable_thinking: bool = False) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _vllm_chat_one(
    handle: VLLMHandle,
    messages: list[dict],
    max_new_tokens: int,
    temperature: float,
    enable_thinking: bool,
    num_return_sequences: int,
) -> list[str]:
    url = f"{handle.base_url}/v1/chat/completions"
    cur_msgs = [dict(m) for m in messages]
    wants_json = "json" in (cur_msgs[0].get("content", "") + " " + cur_msgs[-1].get("content", "")).lower()
    use_json_mode = wants_json
    for attempt in range(3):
        payload = {
            "model": handle.served_model,
            "messages": cur_msgs,
            "max_tokens": int(max_new_tokens),
            "temperature": float(temperature) if temperature and temperature > 0 else 0.0,
            "n": int(num_return_sequences),
            "chat_template_kwargs": {"enable_thinking": bool(enable_thinking)},
        }
        if use_json_mode:
            payload["response_format"] = {"type": "json_object"}
        if temperature and temperature > 0:
            payload["top_p"] = 0.9
        req = urllib.request.Request(
            url, data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
            headers={"Content-Type": "application/json"}, method="POST",
        )
        try:
            with _LOCAL_OPENER.open(req, timeout=300) as resp:
                data = json.loads(resp.read().decode("utf-8"))
            out = []
            for ch in data.get("choices", []):
                txt = (ch.get("message") or {}).get("content") or ""
                out.append(re.sub(r"<think>.*?</think>", "", txt, flags=re.S).strip())
            while len(out) < num_return_sequences:
                out.append("")
            return out[:num_return_sequences]
        except urllib.error.HTTPError as he:
            body = he.read().decode("utf-8", errors="ignore")
            if he.code == 400 and use_json_mode and "response_format" in body.lower():
                use_json_mode = False
                continue
            if he.code == 400 and ("maximum context length" in body.lower() or "max_tokens" in body.lower()):
                for m in cur_msgs:
                    if m.get("role") == "user" and len(m.get("content", "")) > 6000:
                        m["content"] = m["content"][: int(len(m["content"]) * 0.78)]
                continue
            if attempt == 2:
                print(f"  [vLLM HTTP {he.code}] {body[:160]}", flush=True)
                return [""] * num_return_sequences
            time.sleep(1.5 * (attempt + 1))
        except Exception as exc:
            if attempt == 2:
                print(f"  [vLLM ERR] {exc}", flush=True)
                return [""] * num_return_sequences
            time.sleep(1.5 * (attempt + 1))
    return [""] * num_return_sequences


def _run_microbatch_safe(
    model,
    tokenizer,
    prompts: list[str],
    max_new_tokens: int,
    temperature: float,
    num_return_sequences: int = 1,
    max_input_length: int = 5600,
) -> list[list[str]]:
    '''PyTorch 폴백 전용 안전 마이크로배치 (CUDA OOM 시 자동 분할).'''
    if not prompts:
        return []
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    try:
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=max_input_length).to(model.device)
        prompt_len = enc["input_ids"].shape[1]
        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "pad_token_id": tokenizer.pad_token_id,
            "num_return_sequences": num_return_sequences,
            "use_cache": True,
        }
        if temperature and temperature > 0:
            gen_kwargs.update({"do_sample": True, "temperature": float(temperature), "top_p": 0.9})
        else:
            gen_kwargs.update({"do_sample": False, "temperature": None, "top_p": None, "top_k": None})
        with torch.inference_mode():
            out_ids = model.generate(**enc, **gen_kwargs)
        gen_tokens = out_ids[:, prompt_len:]
        decoded = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        cleaned = [re.sub(r"<think>.*?</think>", "", txt, flags=re.S).strip() for txt in decoded]
        grouped = [cleaned[i * num_return_sequences:(i + 1) * num_return_sequences] for i in range(len(prompts))]
        del enc, out_ids
        return grouped
    except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
        if not isinstance(exc, torch.cuda.OutOfMemoryError) and "out of memory" not in str(exc).lower():
            raise
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if len(prompts) == 1:
            if num_return_sequences > 1:
                return [[_run_microbatch_safe(model, tokenizer, prompts, max_new_tokens, temperature, 1, max_input_length)[0][0] for _ in range(num_return_sequences)]]
            if max_input_length > 4096:
                return _run_microbatch_safe(model, tokenizer, prompts, max_new_tokens, temperature, 1, 4096)
            raise
        mid = len(prompts) // 2
        print(f"[OOM-SAFE] batch {len(prompts)} -> split ({mid}, {len(prompts) - mid})", flush=True)
        return (
            _run_microbatch_safe(model, tokenizer, prompts[:mid], max_new_tokens, temperature, num_return_sequences, max_input_length)
            + _run_microbatch_safe(model, tokenizer, prompts[mid:], max_new_tokens, temperature, num_return_sequences, max_input_length)
        )
    finally:
        tokenizer.padding_side = prev_side


def generate_chat_batch(
    model,
    tokenizer,
    messages_list: list[list[dict]],
    max_new_tokens: int = 768,
    batch_size: int = 12,
    temperature: float = 0.0,
    enable_thinking: bool = False,
    num_return_sequences: int = 1,
    verbose: bool = True,
) -> list:
    '''vLLM 동시 스트리밍 배치 추론 (PyTorch SDPA 자동 폴백 포함).'''
    if not messages_list:
        return []
    if isinstance(model, VLLMHandle):
        t0 = time.perf_counter()
        workers = max(1, min(int(batch_size), len(messages_list), 24))
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futs = [
                ex.submit(_vllm_chat_one, model, m, max_new_tokens, temperature, enable_thinking, num_return_sequences)
                for m in messages_list
            ]
            raw_grouped = [f.result() for f in futs]
        dt = time.perf_counter() - t0
        if verbose:
            print(f"  [vLLM] {len(messages_list)}x{num_return_sequences} 완료 | {dt:.1f}s ({dt/max(len(messages_list),1):.2f}s/q, workers={workers})", flush=True)
        return [r[0] if num_return_sequences == 1 else r for r in raw_grouped]

    # PyTorch 폴백 경로 (VRAM 안전 배치 상한 4 적용)
    pt_bs = max(1, min(int(batch_size), 4 if num_return_sequences == 1 else 2))
    prompts = [_render_prompt(tokenizer, m, enable_thinking=enable_thinking) for m in messages_list]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    results = [None] * len(prompts)
    n_batches = (len(order) + pt_bs - 1) // pt_bs
    t_loop = time.perf_counter()

    for b_idx, start in enumerate(range(0, len(order), pt_bs), start=1):
        t_b = time.perf_counter()
        idxs = order[start:start + pt_bs]
        sub_prompts = [prompts[i] for i in idxs]
        outs = _run_microbatch_safe(model, tokenizer, sub_prompts, max_new_tokens, temperature, num_return_sequences)
        for i, res in zip(idxs, outs):
            results[i] = res[0] if num_return_sequences == 1 else res
        if verbose and (b_idx == 1 or b_idx == n_batches or b_idx % max(1, min(3, n_batches)) == 0):
            dt = time.perf_counter() - t_b
            elapsed = time.perf_counter() - t_loop
            eta_m = ((elapsed / b_idx) * (n_batches - b_idx)) / 60.0
            vram_g = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
            print(f"  [PyTorch {b_idx}/{n_batches}] bs={len(sub_prompts)}x{num_return_sequences} | {dt:.1f}s | VRAM={vram_g:.1f}G | ETA={eta_m:.1f}m", flush=True)
    return results


def translate_ko_queries_batch(model, tokenizer, ko_queries: list[str], batch_size: int = 12) -> list[str]:
    if not ko_queries:
        return []
    msgs_list = [
        [
            {"role": "system", "content": "Translate the Korean research-paper question into concise academic English for retrieval. Preserve all acronyms, model names, symbols, and numbers verbatim. Output ONLY the English question."},
            {"role": "user", "content": q},
        ]
        for q in ko_queries
    ]
    raws = generate_chat_batch(model, tokenizer, msgs_list, max_new_tokens=128, batch_size=batch_size)
    out = []
    for q, r in zip(ko_queries, raws):
        lines = [ln.strip() for ln in r.splitlines() if ln.strip()]
        out.append(lines[0] if lines else q)
    return out


def build_v2_grounded_messages(question: str, evidence: list[dict], language: str) -> list[dict]:
    target_lang = "Korean" if language == "ko" else "English"
    ev_block = "\n".join(
        json.dumps({
            "evidence_id": item["evidence_id"], "page": item["page"],
            "section": item["section"], "text": item["text"],
        }, ensure_ascii=False)
        for item in evidence
    )
    system_msg = f'''You are a strict evidence-grounded scientific paper Q&A assistant.

Rules:
1. Rely ONLY on the supplied Evidence JSON lines. NEVER use outside knowledge.
2. First check whether the question contains an explicit or implicit premise that is CONTRADICTED by the evidence (e.g., claiming a method eliminates a bias floor when the paper states it does NOT, or assuming models were trained identically when they were NOT).
   - If contradicted -> set "premise_check": "contradicted", "claims": [], and explain the exact contradiction in "premise_reason" ({target_lang}) citing the contradicting fact.
   - If the evidence does not supply the requested information -> set "premise_check": "unsupported", "claims": [], and explain briefly in "premise_reason" ({target_lang}).
   - Otherwise -> set "premise_check": "supported" and write complete, faithful claims in "claims".
3. Write each claim "text" in natural, accurate academic {target_lang}. Preserve ALL exact numbers, ranges (e.g., 50-225 MHz), units, percentages, dataset/model names, conditions, and qualifiers verbatim.
4. Cover ALL requested items/aspects in the question (lists, comparisons, reasons, caveats, quantitative metrics).
5. Every claim MUST include a valid "evidence_id" from the supplied list and a short "verbatim_quote" copied directly from that evidence text.
6. Output ONLY a single valid JSON object with keys: "premise_check", "premise_reason", "claims". No Markdown code fences.'''

    user_msg = f'''Question ({target_lang}):
{question.strip()}

Evidence:
{ev_block}

Return ONLY JSON matching:
{{"premise_check": "supported|unsupported|contradicted", "premise_reason": "...", "claims": [{{"text": "...", "evidence_id": "ev-01-...", "verbatim_quote": "..."}}]}}'''
    return [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}]


def _extract_json_obj(raw: str) -> dict:
    cleaned = raw.strip()
    m = re.match(r"^```(?:json)?\s*(\{.*\})\s*```$", cleaned, flags=re.I | re.S)
    if m:
        cleaned = m.group(1)
    else:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start != -1 and end > start:
            cleaned = cleaned[start:end + 1]
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return json.loads(cleaned, strict=False)


def _quote_in_source(quote: str, source_text: str) -> bool:
    if not quote or len(quote.strip()) < 8:
        return False
    q_norm = re.sub(r"\s+", " ", quote.casefold()).strip()
    s_norm = re.sub(r"\s+", " ", source_text.casefold()).strip()
    if q_norm in s_norm:
        return True
    q_tokens = re.findall(r"[a-z0-9]+", q_norm)
    if not q_tokens:
        return False
    s_tokens = set(re.findall(r"[a-z0-9]+", s_norm))
    return (sum(t in s_tokens for t in q_tokens) / len(q_tokens)) >= 0.72


def _extract_canonical_numbers(text: str) -> set[str]:
    nums = set()
    cleaned = text.replace("–", "-").replace("−", "-")
    for raw in NUM_PATTERN.findall(cleaned):
        val = raw.lstrip("+").replace(",", "")
        if val.startswith("."):
            val = "0" + val
        try:
            f = float(val)
            nums.add(f"{f:g}")
            nums.add(f"{abs(f):g}")
        except ValueError:
            nums.add(val)
    return nums


def _numeric_fidelity_ok(claim_text: str, source_text: str) -> bool:
    claim_nums = _extract_canonical_numbers(claim_text)
    if not claim_nums:
        return True
    src_nums = _extract_canonical_numbers(source_text)
    strict_nums = {n for n in claim_nums if ("." in n or "-" in n or abs(float(n)) >= 10)}
    return all(n in src_nums for n in strict_nums) if strict_nums else True


def verify_and_assemble_v2(
    raw_output: str,
    evidence: list[dict],
    language: str,
    pre_abstain_reason: str | None = None,
    strict_drop_unverified_numeric: bool = False,
) -> dict:
    ev_by_id = {e["evidence_id"]: e for e in evidence}
    if pre_abstain_reason:
        return {
            "response": canonical_abstention_payload(language, pre_abstain_reason),
            "cited_evidence": [], "retrieved_evidence": evidence, "answer_alignment": [],
            "pipeline": {"abstention_source": "pre_gate", "fallback_used": False, "quote_verified": True, "numeric_verified": True, "raw_verdict": "pre_abstain"},
        }
    try:
        parsed = _extract_json_obj(raw_output)
    except Exception as exc:
        reason = "구조화 JSON 응답 파싱 실패" if language == "ko" else f"JSON parse failed: {exc}"
        return {
            "response": canonical_abstention_payload(language, reason),
            "cited_evidence": [], "retrieved_evidence": evidence, "answer_alignment": [],
            "pipeline": {"abstention_source": "json_fallback", "fallback_used": True, "quote_verified": False, "numeric_verified": False},
        }

    verdict = str(parsed.get("premise_check", "unsupported")).strip().lower()
    reason = str(parsed.get("premise_reason") or "").strip()

    if verdict in {"unsupported", "contradicted"}:
        default_r = "질문의 전제가 논문 근거와 모순되거나 근거가 부족합니다." if language == "ko" else "The question premise is contradicted or unsupported by the paper evidence."
        return {
            "response": canonical_abstention_payload(language, (reason or default_r)[:900]),
            "cited_evidence": [], "retrieved_evidence": evidence, "answer_alignment": [],
            "pipeline": {"abstention_source": f"llm_{verdict}", "fallback_used": False, "quote_verified": True, "numeric_verified": True, "raw_verdict": verdict},
        }

    claims = parsed.get("claims") or []
    verified_texts, cited_ids, alignments = [], [], []
    quote_hits, num_hits = 0, 0
    all_ev_text = " ".join(e["text"] for e in evidence)

    for cl in claims:
        if not isinstance(cl, dict):
            continue
        text = str(cl.get("text", "")).strip()
        eid = str(cl.get("evidence_id", "")).strip()
        quote = str(cl.get("verbatim_quote", "")).strip()
        if not text or eid not in ev_by_id:
            continue
        src = ev_by_id[eid]
        q_ok = _quote_in_source(quote, src["text"])
        n_ok = _numeric_fidelity_ok(text, src["text"]) or _numeric_fidelity_ok(text, all_ev_text)
        if strict_drop_unverified_numeric and not n_ok:
            continue
        quote_hits += int(q_ok)
        num_hits += int(n_ok)
        verified_texts.append(text)
        if eid not in cited_ids and len(cited_ids) < 5:
            cited_ids.append(eid)
        alignments.append({
            "answer_text": text, "evidence_id": eid, "page": src["page"],
            "section": src["section"], "verbatim_quote": quote,
            "quote_verified": q_ok, "numeric_verified": n_ok,
        })

    if not verified_texts or not cited_ids:
        fb = "검증 가능한 인용 근거가 포함된 답변을 구성하지 못했습니다." if language == "ko" else "No verified cited claims were produced."
        return {
            "response": canonical_abstention_payload(language, fb),
            "cited_evidence": [], "retrieved_evidence": evidence, "answer_alignment": [],
            "pipeline": {"abstention_source": "claim_verification_fallback", "fallback_used": True, "quote_verified": False, "numeric_verified": False},
        }

    answer_text = " ".join(verified_texts)[:3950]
    cited_evidence = [ev_by_id[eid] for eid in cited_ids]
    response = {"answer": answer_text, "evidence_ids": cited_ids, "sufficiency": "sufficient", "abstention_reason": None}
    try:
        validate_answer_payload(response, cited_evidence, language)
    except AnswerValidationError as exc:
        fb = f"응답 계약 검증 실패: {exc}" if language == "ko" else f"Contract validation failed: {exc}"
        return {
            "response": canonical_abstention_payload(language, fb[:900]),
            "cited_evidence": [], "retrieved_evidence": evidence, "answer_alignment": [],
            "pipeline": {"abstention_source": "contract_fallback", "fallback_used": True, "quote_verified": False, "numeric_verified": False},
        }

    return {
        "response": response, "cited_evidence": cited_evidence,
        "retrieved_evidence": evidence, "answer_alignment": alignments,
        "pipeline": {
            "abstention_source": None, "fallback_used": False, "raw_verdict": verdict,
            "quote_verified": (quote_hits == len(alignments)),
            "quote_verified_ratio": round(quote_hits / max(len(alignments), 1), 3),
            "numeric_verified": (num_hits == len(alignments)),
            "numeric_verified_ratio": round(num_hits / max(len(alignments), 1), 3),
        },
    }


print(f"[OK] v5.0 초고속 격리 vLLM 추론 엔진 (+ PyTorch 자동 폴백) 준비 완료")

In [ ]:
# Cell 3.2 — 배치 기반 공용 평가 실행기 (`run_pipeline_eval_batched` — 기존 대비 5~6배 고속!)
def run_pipeline_eval(
    model,
    tokenizer,
    exp_name: str,
    split: str = "dev",
    questions: list[dict] | None = None,
    batch_size: int = BATCH_SIZE_4B,
    use_ko_to_en: bool = True,
    use_parent_window: bool = True,
    pre_score_threshold: float = -3.0,
    nli_scorer=None,
    nli_contra_threshold: float = 0.70,
    nli_post_entail_min: float | None = None,
    notes: str = "",
):
    t_all = time.perf_counter()
    set_encoders_device(DEVICE)
    target_qs = questions if questions is not None else [q for q in QUESTIONS if split == "all" or q["split"] == split]
    ckpt_path = RUNS_DIR / f"{exp_name}_{split}.checkpoint.jsonl"
    done = {r["case_id"]: r for r in read_jsonl(ckpt_path)} if ckpt_path.exists() else {}
    pending = [q for q in target_qs if q["case_id"] not in done]
    if done:
        print(f"[RESUME {exp_name}] 기존 체크포인트 {len(done)}/{len(target_qs)}건 로드 (남은 {len(pending)}건)")

    if pending:
        # 1) 한국어 쿼리 일괄 배치 영문 변환 (1회 패스!)
        en_q_map = {}
        if use_ko_to_en:
            ko_pending = [q for q in pending if q["language"] == "ko"]
            if ko_pending:
                en_list = translate_ko_queries_batch(model, tokenizer, [q["question"] for q in ko_pending], batch_size=max(8, batch_size))
                for q, eq in zip(ko_pending, en_list):
                    en_q_map[q["case_id"]] = eq

        # 2) 검색·재정렬 + Pre-Gate NLI 일괄 처리
        need_gen = []
        pre_meta = {}
        for q in pending:
            cid = q["case_id"]
            en_q = en_q_map.get(cid)
            ret = retrieve_v2(q["question"], q["paper_id"], en_query=en_q, top_k=5, use_parent_window=use_parent_window)
            nli_ent, nli_con, pre_reason = 0.5, 0.0, None
            if ret["top_reranker_score"] < pre_score_threshold:
                pre_reason = (
                    "검색된 논문 근거의 관련도가 매우 낮아 답변할 수 없습니다."
                    if q["language"] == "ko"
                    else "Retrieved paper evidence relevance is too low to support an answer."
                )
            elif nli_scorer is not None and ret["evidence"]:
                nli_res = nli_scorer.score_question_against_evidence(q["question"], ret["evidence"])
                nli_ent, nli_con = nli_res["max_entail"], nli_res["max_contra"]
                if nli_con >= nli_contra_threshold and nli_con > nli_ent:
                    pre_reason = (
                        f"질문의 전제가 논문 원문 근거와 모순됩니다 (Scientific NLI 모순 확률={nli_con:.2f})."
                        if q["language"] == "ko"
                        else f"The question premise contradicts the retrieved paper evidence (NLI contradiction={nli_con:.2f})."
                    )
            pre_meta[cid] = {"q": q, "en_q": en_q, "ret": ret, "nli_ent": nli_ent, "nli_con": nli_con, "pre_reason": pre_reason}
            if pre_reason is None:
                need_gen.append((cid, build_v2_grounded_messages(q["question"], ret["evidence"], q["language"])))

        # 3) PyTorch 폴백일 때만 인코더를 CPU로 내림 (vLLM은 기동 시 VRAM이 이미 고정되어 있으므로 이동 불필요!)
        if not isinstance(model, VLLMHandle):
            set_encoders_device("cpu")
        t_gen = time.perf_counter()

        def _finalize_and_write(stream, q_item, raw_str: str | None, wall_s: float):
            cid = q_item["case_id"]
            m = pre_meta[cid]
            ret = m["ret"]
            if m["pre_reason"]:
                out = verify_and_assemble_v2("", ret["evidence"], q_item["language"], pre_abstain_reason=m["pre_reason"])
                if nli_scorer is not None and m["nli_con"] >= nli_contra_threshold:
                    out["pipeline"]["abstention_source"] = "scientific_nli_contradiction"
            else:
                out = verify_and_assemble_v2(raw_str or "", ret["evidence"], q_item["language"])
                if nli_scorer is not None and nli_post_entail_min is not None and out["response"]["sufficiency"] == "sufficient":
                    ev_by_id = {e["evidence_id"]: e for e in ret["evidence"]}
                    kept_al, kept_texts, kept_ids = [], [], []
                    for al in out["answer_alignment"]:
                        sc = nli_scorer.score_focused_claim(ev_by_id[al["evidence_id"]]["text"], al.get("verbatim_quote", ""), al["answer_text"])
                        al["nli_entail"], al["nli_contra"] = round(sc["entail"], 4), round(sc["contra"], 4)
                        if sc["entail"] >= nli_post_entail_min and sc["contra"] < 0.70:
                            kept_al.append(al)
                            kept_texts.append(al["answer_text"])
                            if al["evidence_id"] not in kept_ids:
                                kept_ids.append(al["evidence_id"])
                    if kept_al:
                        out["answer_alignment"] = kept_al
                        out["response"]["answer"] = " ".join(kept_texts)[:3950]
                        out["response"]["evidence_ids"] = kept_ids[:5]
                        out["cited_evidence"] = [ev_by_id[eid] for eid in kept_ids[:5]]
                    else:
                        fb = "생성된 주장이 사후 NLI 근거 검증을 통과하지 못했습니다." if q_item["language"] == "ko" else "Generated claims failed post-hoc NLI entailment verification."
                        out["response"] = canonical_abstention_payload(q_item["language"], fb)
                        out["cited_evidence"] = []
                        out["answer_alignment"] = []
                        out["pipeline"]["abstention_source"] = "post_nli_rejection"
            out["pipeline"].update({
                "top_reranker_score": ret["top_reranker_score"],
                "score_margin_1_5": ret["score_margin_1_5"],
                "en_query": m["en_q"],
                "nli_max_entailment": round(m["nli_ent"], 4),
                "nli_max_contradiction": round(m["nli_con"], 4),
            })
            rec = {"case_id": cid, "output": out, "wall_seconds": round(wall_s, 3)}
            done[cid] = rec
            stream.write(json.dumps(rec, ensure_ascii=False) + "\n")

        with ckpt_path.open("a", encoding="utf-8") as stream:
            for q in pending:
                if pre_meta[q["case_id"]]["pre_reason"] is not None:
                    _finalize_and_write(stream, q, None, 0.2)
            stream.flush()

            need_gen.sort(key=lambda item: sum(len(x["content"]) for x in item[1]))
            chunk_sz = max(batch_size * 2, 24) if isinstance(model, VLLMHandle) else max(batch_size * 2, 8)
            gen_max_tok = 1024 if isinstance(model, VLLMHandle) else 640
            for c_start in range(0, len(need_gen), chunk_sz):
                sub_gen = need_gen[c_start:c_start + chunk_sz]
                t_c = time.perf_counter()
                raws = generate_chat_batch(model, tokenizer, [m for _, m in sub_gen], max_new_tokens=gen_max_tok, batch_size=batch_size, verbose=True)
                raw_map = {}
                broken = []
                for (cid, _), raw in zip(sub_gen, raws):
                    raw_map[cid] = raw
                    try:
                        _extract_json_obj(raw)
                    except Exception:
                        broken.append((cid, raw))
                if broken:
                    print(f"  [1-shot JSON repair] {len(broken)}건 복구 중...", flush=True)
                    rep_msgs = [
                        [
                            {"role": "system", "content": "Fix the malformed JSON below so it is strictly valid JSON with keys premise_check, premise_reason, claims. Do NOT change factual meaning. Output ONLY raw JSON."},
                            {"role": "user", "content": r[:3500]},
                        ]
                        for _, r in broken
                    ]
                    repaired = generate_chat_batch(model, tokenizer, rep_msgs, max_new_tokens=gen_max_tok, batch_size=batch_size, verbose=False)
                    for (cid, _), fixed in zip(broken, repaired):
                        raw_map[cid] = fixed
                per_q = (time.perf_counter() - t_c) / max(len(sub_gen), 1)
                for cid, _ in sub_gen:
                    _finalize_and_write(stream, pre_meta[cid]["q"], raw_map[cid], per_q)
                stream.flush()
                print(f"[{exp_name}] 진행 {len(done)}/{len(target_qs)}건 저장 완료 (최근 {per_q:.2f}s/q)", flush=True)

        gen_per_q = (time.perf_counter() - t_gen) / max(len(pending), 1)
        print(f"[{exp_name}] {len(pending)}건 배치 평가 완료 (총 {time.perf_counter()-t_all:.1f}s, 평균 {gen_per_q:.2f}s/q)", flush=True)

    records = [done[q["case_id"]] for q in target_qs]
    report = evaluate_predictions(records, split=split, questions=target_qs)
    log_experiment(exp_name, report, notes=notes)
    return records, report


def inspect_failures(records: list[dict], report: dict, max_show: int = 5):
    rec_map = {r["case_id"]: r for r in records}
    ev_map = {r["case_id"]: r for r in report["records"]}
    print("=== 1) False-Premise 놓친 사례 ===")
    cnt = 0
    for q in DEV_QUESTIONS:
        ev = ev_map.get(q["case_id"])
        if ev and q["answerability"] == "false_premise" and ev["answered"]:
            out = rec_map[q["case_id"]]["output"]
            print(f"- {q['case_id']} | top={out['pipeline'].get('top_reranker_score', 0):.2f} | nli_con={out['pipeline'].get('nli_max_contradiction', 0):.2f} | {out['response']['answer'][:150]}")
            cnt += 1
            if cnt >= max_show:
                break
    print("=== 2) Answerable 거절 사례 ===")
    cnt = 0
    for q in DEV_QUESTIONS:
        ev = ev_map.get(q["case_id"])
        if ev and q["answerability"] == "answerable" and not ev["answered"]:
            out = rec_map[q["case_id"]]["output"]
            print(f"- {q['case_id']} | src={out['pipeline'].get('abstention_source')} | {str(out['response'].get('abstention_reason'))[:120]}")
            cnt += 1
            if cnt >= max_show:
                break


def _ckpt_is_complete(exp_name: str, split: str, expected_qs: list[dict]) -> bool:
    ckpt_path = RUNS_DIR / f"{exp_name}_{split}.checkpoint.jsonl"
    if not ckpt_path.exists():
        return False
    done_ids = {r["case_id"] for r in read_jsonl(ckpt_path)}
    return all(q["case_id"] in done_ids for q in expected_qs)


def _vllm_is_alive(handle) -> bool:
    if not isinstance(handle, VLLMHandle):
        return False
    try:
        with _LOCAL_OPENER.open(f"{handle.base_url}/health", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False


print("[OK] 배치 기반 공용 평가 실행기 + 스마트 고속 이어하기 준비 완료")

In [ ]:
# Cell 3.3 — [Exp 1C] Base `Qwen/Qwen3-4B-Instruct-2507` Zero-Shot DEV 120문항 배치 평가 (~4분, 완료 시 0.1초 스킵)
_exp1c_name = "exp1c_bge_m3_qwen3_4b_2507_zeroshot"
if _ckpt_is_complete(_exp1c_name, "dev", DEV_QUESTIONS):
    print(f"[FAST-RESUME] {_exp1c_name} 120/120 완료 캐시 사용 (vLLM 기동 생략!)", flush=True)
    llm_4b, tok_4b = None, None
else:
    cleanup_all_gpu_llms()
    llm_4b, tok_4b = load_hf_llm(BASE_STUDENT_MODEL, load_in_4bit=False)

exp1c_records, exp1c_report = run_pipeline_eval(
    llm_4b, tok_4b,
    exp_name=_exp1c_name,
    split="dev",
    batch_size=BATCH_SIZE_4B,
    use_ko_to_en=True,
    use_parent_window=True,
    pre_score_threshold=-3.0,
    notes="Phase 1C baseline: BGE-M3 + Dual-Score Rerank + Parent(1800) + Qwen3-4B-Instruct-2507",
)

compare_experiments_statistically(V1_DEV_REPORT, exp1c_report)
inspect_failures(exp1c_records, exp1c_report)

In [ ]:
# Cell 3.4 — [Exp 1C Teacher Ceiling] `Qwen/Qwen3-14B-AWQ` DEV 상한선 측정 (~7분, 완료 시 0.1초 스킵)
_exp1c_t_name = "exp1c_teacher_qwen3_14b_4bit"
if RUN_TEACHER_BASELINE:
    if _ckpt_is_complete(_exp1c_t_name, "dev", DEV_QUESTIONS):
        print(f"[FAST-RESUME] {_exp1c_t_name} 120/120 완료 캐시 사용 (vLLM 기동 생략!)", flush=True)
        llm_14b, tok_14b = None, None
    else:
        cleanup_all_gpu_llms()
        llm_14b, tok_14b = load_hf_llm(TEACHER_MODEL_NAME, load_in_4bit=True)
    t_records, t_report = run_pipeline_eval(
        llm_14b, tok_14b,
        exp_name=_exp1c_t_name,
        split="dev",
        batch_size=BATCH_SIZE_14B,
        notes="14B Teacher ceiling on DEV",
    )
    compare_experiments_statistically(exp1c_report, t_report)
else:
    print("RUN_TEACHER_BASELINE=False")

In [ ]:
# Cell 3.5 — Batched Semantic Rubric Judge (`Qwen3-14B`, bs=8) + 인간 감사 CSV 내보내기 (~3분)
def run_semantic_judge(records: list[dict], judge_model, judge_tok, label: str, batch_size: int = BATCH_SIZE_14B, export_human_csv: bool = True) -> dict:
    judged, to_grade = [], []
    for rec in records:
        q = QUESTION_MAP[rec["case_id"]]
        if q["answerability"] != "answerable":
            continue
        out = rec["output"]
        ans = out["response"].get("answer", "")
        if out["response"]["sufficiency"] != "sufficient":
            judged.append({"case_id": q["case_id"], "meaning_id": q["meaning_id"], "lang": q["language"], "completeness": 0.0, "faithfulness": 1.0, "reason": "abstained", "question": q["question"], "answer": ans, "cited_pages": []})
            continue
        req_desc = "; ".join("/".join(f["any_of"]) for f in q["required_facts"])
        ev_text = "\n".join(f"[{e['evidence_id']} p.{e['page']}] {e['text'][:800]}" for e in out.get("cited_evidence", []))
        msgs = [
            {"role": "system", "content": '''You are a strict scientific QA grader. Score the Answer on two 0.0-1.0 metrics:
- "completeness": fraction of Required Concepts semantically covered (accept accurate Korean translations/paraphrases!).
- "faithfulness": 1.0 if every factual claim in Answer is directly supported by Cited Evidence (0.0 if hallucinated/contradicted).
Output ONLY JSON: {"completeness": 0.0, "faithfulness": 0.0, "reason": "short note"}'''},
            {"role": "user", "content": f"Question: {q['question']}\nRequired Concepts: {req_desc}\nCited Evidence:\n{ev_text}\nAnswer: {ans}"},
        ]
        to_grade.append((q, out, ans, msgs))

    if to_grade:
        raws = generate_chat_batch(judge_model, judge_tok, [x[3] for x in to_grade], max_new_tokens=256, batch_size=batch_size, enable_thinking=False)
        for (q, out, ans, _), raw in zip(to_grade, raws):
            try:
                j = _extract_json_obj(raw)
                judged.append({
                    "case_id": q["case_id"], "meaning_id": q["meaning_id"], "lang": q["language"],
                    "completeness": float(j.get("completeness", 0.0)), "faithfulness": float(j.get("faithfulness", 0.0)),
                    "reason": str(j.get("reason", ""))[:200], "question": q["question"], "answer": ans,
                    "cited_pages": [e["page"] for e in out.get("cited_evidence", [])],
                })
            except Exception as exc:
                judged.append({"case_id": q["case_id"], "meaning_id": q["meaning_id"], "lang": q["language"], "completeness": 0.0, "faithfulness": 0.0, "reason": f"judge_err:{exc}", "question": q["question"], "answer": ans, "cited_pages": []})

    by_lang = {l: [x for x in judged if x["lang"] == l] for l in ("en", "ko")}
    by_mid = defaultdict(list)
    for x in judged:
        by_mid[x["meaning_id"]].append(x)
    _mean = lambda vals: round(statistics.mean(vals), 3) if vals else 0.0
    paired_vals = [min(g[0]["completeness"], g[1]["completeness"]) for g in by_mid.values() if len(g) == 2]
    summary = {
        "en_completeness": _mean([x["completeness"] for x in by_lang["en"]]),
        "ko_completeness": _mean([x["completeness"] for x in by_lang["ko"]]),
        "paired_completeness": _mean(paired_vals),
        "mean_faithfulness": _mean([x["faithfulness"] for x in judged]),
    }
    (RUNS_DIR / f"{label}_semantic_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"[SEMANTIC JUDGE — {label}]", json.dumps(summary, ensure_ascii=False))
    if export_human_csv:
        csv_path = RUNS_DIR / f"{label}_human_audit.csv"
        df = pd.DataFrame(judged)
        df["human_completeness_0_1"] = ""
        df["human_faithful_0_1"] = ""
        df["human_notes"] = ""
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"[EXPORTED] 인간 감사 시트: {csv_path}")
    return summary


if RUN_SEMANTIC_JUDGE:
    _sem_file = RUNS_DIR / "exp1c_qwen3_4b_2507_semantic_summary.json"
    if _sem_file.exists():
        sem_exp1c = json.loads(_sem_file.read_text(encoding="utf-8"))
        print(f"[CACHE HIT] Semantic Judge -> {sem_exp1c}", flush=True)
    else:
        if "llm_14b" not in globals() or not _vllm_is_alive(globals().get("llm_14b")):
            cleanup_all_gpu_llms()
            llm_14b, tok_14b = load_hf_llm(TEACHER_MODEL_NAME, load_in_4bit=True)
        sem_exp1c = run_semantic_judge(exp1c_records, llm_14b, tok_14b, "exp1c_qwen3_4b_2507")
else:
    print("RUN_SEMANTIC_JUDGE=False")

## Section 4. [Phase 2] Scientific NLI / 전제·충실도 검증기 (`mDeBERTa-v3-base` Full FT)

In [ ]:
# Cell 4.1 — 외부 arXiv 전체 본문 150편 수집·청킹 (HF Parquet, 차단 0%, ~10초) + SciFact 로더
import tarfile
import urllib.request

SCIFACT_DIR = DATASETS_DIR / "scifact"
EXT_CHUNKS_PATH = DATASETS_DIR / "external_arxiv_full_chunks.jsonl"
EXT_META_PATH = DATASETS_DIR / "external_arxiv_meta.jsonl"
NLI_SYNTH_PATH = DATASETS_DIR / "synth_fullchunk_premise_nli.jsonl"


def _simple_chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    text = repair_pdf_chunk_text(re.sub(r"-\s*\n", "", text))
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            cut = text.rfind(". ", start + chunk_size // 2, end)
            if cut != -1:
                end = cut + 1
        piece = text[start:end].strip()
        if len(piece) >= 120:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(start + 1, end - overlap)
    return chunks


def load_scifact_pairs() -> list[dict]:
    SCIFACT_DIR.mkdir(parents=True, exist_ok=True)
    tgz = SCIFACT_DIR / "data.tar.gz"
    if not (SCIFACT_DIR / "data/corpus.jsonl").exists():
        urllib.request.urlretrieve("https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", tgz)
        with tarfile.open(tgz, "r:gz") as tar:
            try:
                tar.extractall(SCIFACT_DIR, filter="data")
            except TypeError:
                tar.extractall(SCIFACT_DIR)
    corpus = {r["doc_id"]: " ".join(r["abstract"]) for r in read_jsonl(SCIFACT_DIR / "data/corpus.jsonl")}
    claims = read_jsonl(SCIFACT_DIR / "data/claims_train.jsonl")
    lmap = {"SUPPORT": 0, "NEUTRAL": 1, "CONTRADICT": 2}
    pairs = []
    for cl in claims:
        ev = cl.get("evidence") or {}
        if not ev:
            for doc_id in (cl.get("cited_doc_ids") or [])[:1]:
                if doc_id in corpus:
                    pairs.append({"premise": corpus[doc_id], "hypothesis": cl["claim"], "label": 1, "source": "scifact_nei"})
            continue
        for doc_id_s, items in ev.items():
            did = int(doc_id_s)
            if did not in corpus:
                continue
            for it in items:
                if it["label"] in lmap:
                    pairs.append({"premise": corpus[did], "hypothesis": cl["claim"], "label": lmap[it["label"]], "source": "scifact"})
    print(f"[SciFact] loaded {len(pairs)} pairs: {Counter(p['label'] for p in pairs)}")
    return pairs


def fetch_and_chunk_external_arxiv_pdfs(target_papers: int = EXT_TRAIN_PAPERS, max_chunks_per_paper: int = 24) -> tuple[list[dict], dict[str, list[dict]]]:
    if EXT_META_PATH.exists() and EXT_CHUNKS_PATH.exists():
        metas = read_jsonl(EXT_META_PATH)
        if len(metas) >= target_papers:
            by_p = defaultdict(list)
            for c in read_jsonl(EXT_CHUNKS_PATH):
                by_p[c["paper_id"]].append(c)
            if sum(len(v) for v in by_p.values()) >= target_papers * 8:
                print(f"[CACHE HIT] external full-text papers={len(metas)}, chunks={sum(len(v) for v in by_p.values())}")
                return metas, by_p

    # arXiv HTTP 차단(GCP IP 429) 완전 회피: HuggingFace Parquet 전체 본문 arXiv 스트리밍 (150편 약 10초!)
    from datasets import load_dataset
    EXT_META_PATH.unlink(missing_ok=True)
    EXT_CHUNKS_PATH.unlink(missing_ok=True)
    metas, by_p = [], defaultdict(list)
    seen_hashes = set()

    print(f"[DATA] HuggingFace Parquet(`ccdv/arxiv-summarization`)에서 외부 arXiv 전체 본문 {target_papers}편 수집 중...", flush=True)
    stream_iter = None
    try:
        stream_iter = load_dataset("ccdv/arxiv-summarization", "document", split="train", streaming=True)
    except Exception as exc:
        print(f"[INFO] streaming 실패 -> 직접 Parquet 샤드 로드 ({exc})", flush=True)
        df_pq = pd.read_parquet("https://huggingface.co/datasets/ccdv/arxiv-summarization/resolve/main/document/train-00000-of-00032.parquet")
        stream_iter = df_pq.to_dict(orient="records")

    with EXT_META_PATH.open("w", encoding="utf-8") as mf, EXT_CHUNKS_PATH.open("w", encoding="utf-8") as cf:
        for idx, ex in enumerate(stream_iter):
            if len(metas) >= target_papers:
                break
            abstract = repair_pdf_chunk_text(str(ex.get("abstract") or ""))
            article = repair_pdf_chunk_text(str(ex.get("article") or ""))
            if len(abstract) < 200 or len(article) < 6000:
                continue
            head_norm = abstract[:200].casefold()
            if any(ft in head_norm for ft in FORBIDDEN_TITLES if len(ft) > 20):
                continue
            h_sig = hashlib.md5(head_norm.encode("utf-8")).hexdigest()
            if h_sig in seen_hashes:
                continue

            pid = f"ext-arxiv-{len(metas)+1:04d}"
            body_chunks = _simple_chunk_text(article, chunk_size=1200, overlap=200)[: max_chunks_per_paper - 1]
            if len(body_chunks) < 7:
                continue
            paper_chunks = [{
                "paper_id": pid, "chunk_id": f"{pid}-p001-c001",
                "page": 1, "section": "Abstract", "text": abstract[:1400],
            }]
            for c_i, txt in enumerate(body_chunks, start=1):
                pno = min(15, 1 + (c_i // 2))
                paper_chunks.append({
                    "paper_id": pid, "chunk_id": f"{pid}-p{pno:03d}-c{c_i+1:03d}",
                    "page": pno, "section": "Body", "text": txt,
                })
            meta = {"paper_id": pid, "arxiv_id": f"hf-ccdv-{idx}", "title": abstract[:120], "n_chunks": len(paper_chunks)}
            mf.write(json.dumps(meta, ensure_ascii=False) + "\n")
            for ch in paper_chunks:
                cf.write(json.dumps(ch, ensure_ascii=False) + "\n")
            metas.append(meta)
            by_p[pid] = paper_chunks
            seen_hashes.add(h_sig)
            if len(metas) % 50 == 0:
                print(f"외부 전체 논문 청킹: {len(metas)}/{target_papers}편 완료", flush=True)

    assert len(metas) >= min(50, target_papers), f"외부 논문 수집 부족: {len(metas)}편 (0편 진행 차단!)"
    print(f"[READY] 외부 Full-Text 논문 {len(metas)}편, 총 {sum(len(v) for v in by_p.values())}청크 (corpus30 0% 겹침)", flush=True)
    return metas, by_p


scifact_pairs = load_scifact_pairs()
ext_metas, ext_chunks_by_paper = fetch_and_chunk_external_arxiv_pdfs(target_papers=EXT_TRAIN_PAPERS)

In [ ]:
# Cell 4.2 — Batched Teacher(`Qwen3-14B`, bs=6) 6종 자연 표현 EN/KO NLI 대조쌍 합성 (~35분)
FP_STYLES = [
    "explicit_premise_clause (e.g., 'Since the paper establishes that X, ...' / '논문이 X라고 입증했으므로 ...')",
    "implicit_why_how_presupposition (e.g., 'Why does the proposed method completely eliminate X?' / '제안 방법이 X를 완전히 제거하는 이유는?')",
    "numeric_or_range_flip (flip a concrete number, percentage, or threshold in the passage)",
    "negation_or_caveat_removal (turn a stated limitation or non-requirement into a positive claim)",
    "entity_or_baseline_swap (attribute baseline A's result or property to method B)",
    "overgeneralization (claim a conditional/single-dataset result holds universally without exceptions)",
]

if RUN_NLI_SYNTH:
    done_chunks = {r["chunk_id"] for r in read_jsonl(NLI_SYNTH_PATH)} if NLI_SYNTH_PATH.exists() else set()
    rng = random.Random(378)
    selected_chunks = []
    for meta in ext_metas:
        chs = ext_chunks_by_paper[meta["paper_id"]]
        selected_chunks.extend([chs[0]] + rng.sample(chs[1:], min(3, len(chs) - 1)))
    pending_chunks = [(idx, ch) for idx, ch in enumerate(selected_chunks, start=1) if ch["chunk_id"] not in done_chunks]
    print(f"[NLI SYNTH] 전체 {len(selected_chunks)}청크 중 남은 {len(pending_chunks)}청크 (bs={BATCH_SIZE_14B})", flush=True)

    if pending_chunks and ("llm_14b" not in globals() or not _vllm_is_alive(globals().get("llm_14b"))):
        cleanup_all_gpu_llms()
        llm_14b, tok_14b = load_hf_llm(TEACHER_MODEL_NAME, load_in_4bit=True)

    t0 = time.perf_counter()
    with NLI_SYNTH_PATH.open("a", encoding="utf-8") as stream:
        macro = BATCH_SIZE_14B * 4
        for start in range(0, len(pending_chunks), macro):
            sub = pending_chunks[start:start + macro]
            msgs_list = []
            for idx, ch in sub:
                style = FP_STYLES[idx % len(FP_STYLES)]
                msgs_list.append([
                    {"role": "system", "content": f'''Given a research-paper excerpt, create a JSON object with 6 bilingual verification items:
- "entailed_en", "entailed_ko": a specific claim OR question whose presupposition is DIRECTLY supported by the excerpt.
- "contradicted_en", "contradicted_ko": a realistic FALSE-PREMISE question or claim that contradicts the excerpt using style: {style}.
- "neutral_en", "neutral_ko": a plausible IN-DOMAIN question/claim asking about an experiment detail, hyperparameter, or metric NOT stated in the excerpt.
Output ONLY JSON with those 6 keys.'''},
                    {"role": "user", "content": ch["text"]},
                ])
            raws = generate_chat_batch(
                llm_14b, tok_14b, msgs_list,
                max_new_tokens=1024 if TEACHER_THINKING_SYNTH else 640,
                batch_size=BATCH_SIZE_14B,
                enable_thinking=TEACHER_THINKING_SYNTH,
            )
            for (idx, ch), raw in zip(sub, raws):
                style = FP_STYLES[idx % len(FP_STYLES)]
                try:
                    obj = _extract_json_obj(raw)
                    for k, lbl, lang in [
                        ("entailed_en", 0, "en"), ("entailed_ko", 0, "ko"),
                        ("neutral_en", 1, "en"), ("neutral_ko", 1, "ko"),
                        ("contradicted_en", 2, "en"), ("contradicted_ko", 2, "ko"),
                    ]:
                        hyp = str(obj.get(k, "")).strip()
                        if len(hyp) >= 8:
                            stream.write(json.dumps({
                                "chunk_id": ch["chunk_id"], "paper_id": ch["paper_id"],
                                "premise": ch["text"], "hypothesis": hyp, "label": lbl, "lang": lang, "style": style.split()[0],
                            }, ensure_ascii=False) + "\n")
                except Exception:
                    continue
            stream.flush()
            done_n = min(start + macro, len(pending_chunks))
            print(f"NLI 배치 합성: {done_n}/{len(pending_chunks)} ({(time.perf_counter()-t0)/60:.1f}분 경과)")
else:
    print("RUN_NLI_SYNTH=False")

In [ ]:
# Cell 4.3 — `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` Paper-Split Full Fine-Tuning (L4 ~20분)
NLI_BASE_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
NLI_SAVE_DIR = MODELS_DIR / "scientific_nli_mdeberta"

if RUN_NLI_TRAIN:
    from datasets import Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

    cleanup_all_gpu_llms()
    set_encoders_device("cpu")

    # 새 FP32 NLI 재학습 시 구 Exp 2A/2B 체크포인트 초기화 (깨진 캐시 재사용 방지!)
    for _old_ckpt in ("exp2a_pre_sliding_nli_dev.checkpoint.jsonl", "exp2b_pre_and_2stage_post_nli_dev.checkpoint.jsonl"):
        (RUNS_DIR / _old_ckpt).unlink(missing_ok=True)

    synth_rows = read_jsonl(NLI_SYNTH_PATH) if NLI_SYNTH_PATH.exists() else []
    synth_pids = sorted({r["paper_id"] for r in synth_rows})
    random.Random(378).shuffle(synth_pids)
    val_pids = set(synth_pids[: max(1, len(synth_pids) // 10)])

    tr_rows = list(scifact_pairs) + [r for r in synth_rows if r["paper_id"] not in val_pids] * 2
    va_rows = [r for r in synth_rows if r["paper_id"] in val_pids]
    random.Random(378).shuffle(tr_rows)
    if not va_rows:
        cut = max(1, len(tr_rows) // 10)
        va_rows, tr_rows = tr_rows[:cut], tr_rows[cut:]
    print(f"NLI Train={len(tr_rows)} | Paper-Heldout Val={len(va_rows)} | Dist={Counter(r['label'] for r in tr_rows)}")

    nli_tok = AutoTokenizer.from_pretrained(NLI_BASE_MODEL)
    # transformers 5.x는 기본 dtype='auto'(fp16)로 로드할 수 있으므로 mDeBERTa-v3는 반드시 float32 명시!
    nli_mod = AutoModelForSequenceClassification.from_pretrained(NLI_BASE_MODEL, num_labels=3, torch_dtype=torch.float32)

    def _tok(batch):
        return nli_tok(batch["premise"], batch["hypothesis"], truncation=True, max_length=512)

    tr_ds = Dataset.from_list(tr_rows).map(_tok, batched=True, remove_columns=list(tr_rows[0].keys())).add_column("labels", [r["label"] for r in tr_rows])
    va_ds = Dataset.from_list(va_rows).map(_tok, batched=True, remove_columns=list(va_rows[0].keys())).add_column("labels", [r["label"] for r in va_rows])

    ta_sig = inspect.signature(TrainingArguments.__init__).parameters
    ta_raw = {
        "output_dir": str(NLI_SAVE_DIR / "checkpoints"),
        "learning_rate": 1.5e-5, "warmup_steps": 100, "max_grad_norm": 1.0,
        "per_device_train_batch_size": 16, "per_device_eval_batch_size": 32,
        "num_train_epochs": 3, "weight_decay": 0.01,
        "bf16": False, "fp16": False,
        "eval_strategy": "epoch", "save_strategy": "epoch", "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss", "save_total_limit": 1, "logging_steps": 25, "seed": 378, "report_to": "none",
    }
    args = TrainingArguments(**{k: v for k, v in ta_raw.items() if k in ta_sig})
    t_sig = inspect.signature(Trainer.__init__).parameters
    tk = {"model": nli_mod, "args": args, "train_dataset": tr_ds, "eval_dataset": va_ds}
    tk["processing_class" if "processing_class" in t_sig else "tokenizer"] = nli_tok

    trainer = Trainer(**tk)
    trainer.train()
    NLI_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(NLI_SAVE_DIR))
    nli_tok.save_pretrained(str(NLI_SAVE_DIR))
    shutil.rmtree(NLI_SAVE_DIR / "checkpoints", ignore_errors=True)

    # 자동 건전성 검증 (NaN/클래스 붕괴 차단!)
    nli_mod.eval()
    _chk_pairs = [
        ("The receiver operates across 50-225 MHz.", "The receiver covers 50-225 MHz."),
        ("Method X does NOT eliminate the bias floor.", "Method X completely eliminates the bias floor."),
    ]
    _enc = nli_tok([p[0] for p in _chk_pairs], [p[1] for p in _chk_pairs], padding=True, truncation=True, return_tensors="pt").to(nli_mod.device)
    with torch.no_grad():
        _probs = F.softmax(nli_mod(**_enc).logits, dim=-1).cpu().tolist()
    print(f"[NLI SANITY] entail_pair={_probs[0]} | contra_pair={_probs[1]}", flush=True)
    if any(not np.isfinite(x) for row in _probs for x in row) or _probs[0][0] < 0.5 or _probs[1][2] < 0.5:
        print("[WARN] NLI 건전성 미달 -> 검증된 사전학습 mDeBERTa-v3-mnli-xnli로 안전 복구!", flush=True)
        _safe_mod = AutoModelForSequenceClassification.from_pretrained(NLI_BASE_MODEL)
        _safe_mod.save_pretrained(str(NLI_SAVE_DIR))
        del _safe_mod
    print(f"[SAVED] Scientific NLI 저장 완료: {NLI_SAVE_DIR}")
    nli_mod.cpu()
    free_vram("nli_mod", "trainer", "_enc", "_probs")
else:
    print("RUN_NLI_TRAIN=False")

In [ ]:
# Cell 4.4 — Sliding-Window Pre-NLI + 2-Stage Focused Post-NLI + [Exp 2A / 2B] 배치 DEV 평가 (~8분)
from transformers import AutoModelForSequenceClassification, AutoTokenizer


class SlidingWindowScientificNLI:
    def __init__(self, model_dir: str | Path, device: str = DEVICE):
        actual = device
        if actual == "cuda" and torch.cuda.is_available():
            if (torch.cuda.mem_get_info()[0] / (1024 ** 3)) < 1.5:
                actual = "cpu"
        self.tok = AutoTokenizer.from_pretrained(str(model_dir))
        self.model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), torch_dtype=torch.float32).to(actual).eval()
        self.device = actual
        id2lbl = {int(k): str(v).lower() for k, v in (getattr(self.model.config, "id2label", None) or {}).items()}
        self.idx_entail = next((i for i, v in id2lbl.items() if "entail" in v), 0)
        self.idx_neutral = next((i for i, v in id2lbl.items() if "neutral" in v), 1)
        self.idx_contra = next((i for i, v in id2lbl.items() if "contra" in v), 2)

    @staticmethod
    def _windows(text: str, win_words: int = 200, stride: int = 120) -> list[str]:
        words = text.split()
        if len(words) <= win_words:
            return [text]
        out = []
        for i in range(0, len(words), stride):
            chunk = " ".join(words[i:i + win_words])
            if chunk:
                out.append(chunk)
            if i + win_words >= len(words):
                break
        return out

    @staticmethod
    def _focused_quote_window(evidence_text: str, quote: str, pad_chars: int = 250) -> str | None:
        if not quote or len(quote.strip()) < 8:
            return None
        idx = evidence_text.casefold().find(quote.strip().casefold())
        if idx == -1:
            return None
        return evidence_text[max(0, idx - pad_chars):min(len(evidence_text), idx + len(quote) + pad_chars)].strip()

    @staticmethod
    def _extract_premise_claim(question: str) -> list[str]:
        hyps = [question.strip()]
        m_en = re.search(r"(?:since|given that|assuming that)\s+(?:the paper\s+(?:establishes|proves|shows)\s+that\s+)?([^,?]+)", question, flags=re.I)
        if m_en and len(m_en.group(1).strip()) >= 12:
            hyps.append(m_en.group(1).strip())
        m_ko = re.search(r"(?:논문이|본 연구가)?\s*(.+?)(?:다고|라고)\s*(?:입증했으므로|보였으므로|가정할 때|전제할 때)", question)
        if m_ko and len(m_ko.group(1).strip()) >= 8:
            hyps.append(m_ko.group(1).strip())
        return hyps

    def _predict_batch(self, pairs: list[tuple[str, str]], batch_size: int = 8) -> list[dict]:
        results = []
        i = 0
        cur_bs = batch_size
        while i < len(pairs):
            sub = pairs[i:i + cur_bs]
            try:
                enc = self.tok(
                    [p[0] for p in sub], [p[1] for p in sub],
                    padding=True, truncation=True, max_length=512, return_tensors="pt"
                ).to(self.device)
                with torch.inference_mode():
                    probs = F.softmax(self.model(**enc).logits, dim=-1).cpu().tolist()
                del enc
                for p in probs:
                    if not all(np.isfinite(x) for x in p):
                        results.append({"entail": 0.5, "neutral": 0.5, "contra": 0.0})
                    else:
                        results.append({"entail": p[self.idx_entail], "neutral": p[self.idx_neutral], "contra": p[self.idx_contra]})
                i += len(sub)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if cur_bs > 1:
                    cur_bs = max(1, cur_bs // 2)
                else:
                    self.model.to("cpu")
                    self.device = "cpu"
        return results

    def score_question_against_evidence(self, question: str, evidence: list[dict]) -> dict:
        hyps = self._extract_premise_claim(question)
        pairs = [(w, h) for e in evidence for w in self._windows(e["text"]) for h in hyps]
        if not pairs:
            return {"max_entail": 0.0, "max_contra": 0.0}
        scores = self._predict_batch(pairs)
        return {"max_entail": max(s["entail"] for s in scores), "max_contra": max(s["contra"] for s in scores)}

    def score_focused_claim(self, evidence_text: str, verbatim_quote: str, claim_text: str) -> dict:
        focused = self._focused_quote_window(evidence_text, verbatim_quote, pad_chars=250)
        wins = [focused] if focused else self._windows(evidence_text)
        scores = self._predict_batch([(w, claim_text) for w in wins])
        return {"entail": max(s["entail"] for s in scores), "contra": max(s["contra"] for s in scores)}


if RUN_EXP2_ABLATION and NLI_SAVE_DIR.exists():
    cleanup_all_gpu_llms()
    set_encoders_device(DEVICE)
    llm_4b, tok_4b = load_hf_llm(BASE_STUDENT_MODEL, load_in_4bit=False)
    nli_scorer = SlidingWindowScientificNLI(NLI_SAVE_DIR, device=DEVICE)

    exp2a_records, exp2a_report = run_pipeline_eval(
        llm_4b, tok_4b, exp_name="exp2a_pre_sliding_nli", split="dev",
        batch_size=BATCH_SIZE_4B, nli_scorer=nli_scorer, nli_contra_threshold=0.70, nli_post_entail_min=None,
        notes="Phase 2A: Sliding-Window Scientific NLI Pre-Gate",
    )
    exp2b_records, exp2b_report = run_pipeline_eval(
        llm_4b, tok_4b, exp_name="exp2b_pre_and_2stage_post_nli", split="dev",
        batch_size=BATCH_SIZE_4B, nli_scorer=nli_scorer, nli_contra_threshold=0.70, nli_post_entail_min=0.25,
        notes="Phase 2B: Pre-Gate + 2-Stage Focused Post-NLI",
    )
    compare_experiments_statistically(exp1c_report, exp2b_report)
else:
    print("RUN_EXP2_ABLATION skipped")

## Section 5. [Phase 3] 배치 역합성 5-Chunk SFT (+ QASPER 500) · Completion-Only LoRA · 병렬 On-Policy RPO-DPO

In [ ]:
# Cell 5.1 — Batched Ground-Truth-First 5-Chunk 역합성 (`Qwen3-14B`, bs=6) + QASPER 500 혼합 (~1.5시간)
SFT_DATA_PATH = DATASETS_DIR / "distill_5chunk_reverse_sft_v4.jsonl"

if RUN_QA_SYNTH:
    cleanup_all_gpu_llms()
    llm_14b, tok_14b = load_hf_llm(TEACHER_MODEL_NAME, load_in_4bit=True)
    set_encoders_device(DEVICE)

    done_pids = {r["paper_id"] for r in read_jsonl(SFT_DATA_PATH)} if SFT_DATA_PATH.exists() else set()
    pending_metas = [(idx, m) for idx, m in enumerate(ext_metas, start=1) if m["paper_id"] not in done_pids]
    print(f"[SFT SYNTH] 전체 {len(ext_metas)}편 중 남은 {len(pending_metas)}편 배치 역합성 시작 (workers={BATCH_SIZE_14B})", flush=True)

    t0 = time.perf_counter()
    with SFT_DATA_PATH.open("a", encoding="utf-8") as sft_f:
        macro = min(15, max(8, BATCH_SIZE_14B))  # 12편씩 묶어 질문 생성 -> 검색 -> 답변 배치 생성
        for start in range(0, len(pending_metas), macro):
            sub_metas = pending_metas[start:start + macro]

            # Step 1: 12편의 원자 사실 + 6종 질문을 한 번에 배치 생성
            q_msgs_list = []
            for idx, meta in sub_metas:
                p_chunks = ext_chunks_by_paper[meta["paper_id"]]
                gold_chunks = [p_chunks[0], p_chunks[min(len(p_chunks) - 1, max(1, len(p_chunks) // 2))]]
                gold_text = "\n---\n".join(c["text"] for c in gold_chunks)
                style = FP_STYLES[idx % len(FP_STYLES)]
                q_msgs_list.append([
                    {"role": "system", "content": f'''Read the paper excerpts and return ONE JSON object with:
- "atomic_facts": list of 2-4 short exact keywords/numbers/terms from the excerpts that MUST appear in a complete answer to ans_en.
- "ans_en", "ans_ko": a rich answerable question requiring ALL atomic_facts.
- "unsup_en", "unsup_ko": a plausible IN-DOMAIN question about a metric/ablation/hyperparameter NOT in the paper.
- "fp_en", "fp_ko": a realistic FALSE-PREMISE question contradicting the excerpts using style: {style}.
Output ONLY raw JSON.'''},
                    {"role": "user", "content": gold_text[:2800]},
                ])
            q_raws = generate_chat_batch(
                llm_14b, tok_14b, q_msgs_list,
                max_new_tokens=1152 if TEACHER_THINKING_SYNTH else 768,
                batch_size=BATCH_SIZE_14B, enable_thinking=TEACHER_THINKING_SYNTH,
            )

            # Step 2: 질문 파싱 + 한국어 질문 배치 번역 + 5-Chunk 실제 검색
            qa_jobs = []
            ko_to_tr = []
            paper_indices = {}
            for (_, meta), q_raw in zip(sub_metas, q_raws):
                pid = meta["paper_id"]
                try:
                    q_obj = _extract_json_obj(q_raw)
                except Exception:
                    continue
                atomic_facts = [str(f).strip() for f in (q_obj.get("atomic_facts") or []) if str(f).strip()]
                p_chunks = ext_chunks_by_paper[pid]
                p_embs = embedder.encode([c["text"] for c in p_chunks], batch_size=32, normalize_embeddings=True).astype(np.float32)
                p_bm25 = bm25_retrieval.build_index(p_chunks)
                paper_indices[pid] = (p_chunks, p_embs, p_bm25)

                for key, lang, exp_v in [
                    ("ans_en", "en", "supported"), ("ans_ko", "ko", "supported"),
                    ("unsup_en", "en", "unsupported"), ("unsup_ko", "ko", "unsupported"),
                    ("fp_en", "en", "contradicted"), ("fp_ko", "ko", "contradicted"),
                ]:
                    q_text = str(q_obj.get(key, "")).strip()
                    if len(q_text) < 8:
                        continue
                    job = {"pid": pid, "key": key, "lang": lang, "exp_v": exp_v, "q_text": q_text, "atomic_facts": atomic_facts}
                    qa_jobs.append(job)
                    if lang == "ko":
                        ko_to_tr.append(job)

            if ko_to_tr:
                en_trs = translate_ko_queries_batch(llm_14b, tok_14b, [j["q_text"] for j in ko_to_tr], batch_size=max(8, BATCH_SIZE_14B))
                for j, eq in zip(ko_to_tr, en_trs):
                    j["en_q"] = eq

            # 실제 Top-5 검색 (하드 디스트랙터 포함) -> 답변 배치 생성
            ans_msgs_list = []
            for j in qa_jobs:
                p_chunks, p_embs, p_bm25 = paper_indices[j["pid"]]
                ret = retrieve_from_chunks(j["q_text"], p_chunks, p_embs, p_bm25, en_query=j.get("en_q"), top_k=5, use_parent_window=True)
                j["evidence"] = ret["evidence"]
                j["msgs"] = build_v2_grounded_messages(j["q_text"], ret["evidence"], j["lang"])
                ans_msgs_list.append(j["msgs"])

            ans_raws = generate_chat_batch(
                llm_14b, tok_14b, ans_msgs_list,
                max_new_tokens=1280 if TEACHER_THINKING_SYNTH else 1024,
                batch_size=BATCH_SIZE_14B, enable_thinking=TEACHER_THINKING_SYNTH,
            )

            # Step 3: 원자 사실 + Quote + 수치 3중 자동 필터 통과 샘플만 SFT 저장!
            for j, raw in zip(qa_jobs, ans_raws):
                checked = verify_and_assemble_v2(raw, j["evidence"], j["lang"])
                fact_ok = True
                if j["key"] == "ans_en" and j["atomic_facts"]:
                    ans_low = checked["response"]["answer"].casefold()
                    hits = sum(f.casefold() in ans_low for f in j["atomic_facts"])
                    fact_ok = (hits >= max(1, (len(j["atomic_facts"]) + 1) // 2))
                if (
                    checked["pipeline"].get("raw_verdict") == j["exp_v"]
                    and checked["pipeline"].get("quote_verified")
                    and checked["pipeline"].get("numeric_verified")
                    and fact_ok
                ):
                    clean_json = json.dumps(_extract_json_obj(raw), ensure_ascii=False)
                    sft_f.write(json.dumps({
                        "paper_id": j["pid"], "q_type": j["key"], "language": j["lang"],
                        "expected_verdict": j["exp_v"], "question": j["q_text"],
                        "atomic_facts": j["atomic_facts"], "evidence": j["evidence"],
                        "messages": j["msgs"] + [{"role": "assistant", "content": clean_json}],
                    }, ensure_ascii=False) + "\n")
            sft_f.flush()
            done_n = min(start + macro, len(pending_metas))
            print(f"Reverse 5-Chunk SFT 배치 합성: {done_n}/{len(pending_metas)}편 ({(time.perf_counter()-t0)/60:.1f}분 경과)")

    # QASPER 인간 작성 논문 QA 500건 혼합 (datasets>=4.0 스크립트 차단 대비 Parquet 직접 폴백 포함)
    existing_sft = read_jsonl(SFT_DATA_PATH) if SFT_DATA_PATH.exists() else []
    if MIX_QASPER_SAMPLES > 0 and not any(r.get("paper_id", "").startswith("qasper-") for r in existing_sft):
        try:
            from datasets import load_dataset
            try:
                qds = load_dataset("tau/scrolls", "qasper", split=f"train[:{MIX_QASPER_SAMPLES}]")
            except Exception:
                qds = load_dataset(
                    "parquet",
                    data_files="https://huggingface.co/datasets/tau/scrolls/resolve/refs/convert/parquet/qasper/train/0000.parquet",
                    split=f"train[:{MIX_QASPER_SAMPLES}]",
                )
            added = 0
            with SFT_DATA_PATH.open("a", encoding="utf-8") as sft_f:
                for i, ex in enumerate(qds):
                    lines = ex["input"].splitlines()
                    q_line, body = lines[0].strip(), " ".join(lines[1:])[:1800]
                    out_gt = str(ex["output"]).strip()
                    is_unans = out_gt.lower() in {"unanswerable", "none", ""}
                    ev = [{"evidence_id": f"ev-01-qasper-{i:04d}-p001-c001", "paper_id": f"qasper-{i:04d}", "page": 1, "section": "Paper", "text": body}]
                    msgs = build_v2_grounded_messages(q_line, ev, "en")
                    ans_obj = (
                        {"premise_check": "unsupported", "premise_reason": "The supplied excerpt does not answer this question.", "claims": []}
                        if is_unans
                        else {"premise_check": "supported", "premise_reason": "Supported by paper excerpt.", "claims": [{"text": out_gt, "evidence_id": ev[0]["evidence_id"], "verbatim_quote": body[:160]}]}
                    )
                    sft_f.write(json.dumps({
                        "paper_id": f"qasper-{i:04d}", "q_type": "qasper_unans" if is_unans else "qasper_ans",
                        "language": "en", "expected_verdict": "unsupported" if is_unans else "supported",
                        "question": q_line, "evidence": ev,
                        "messages": msgs + [{"role": "assistant", "content": json.dumps(ans_obj, ensure_ascii=False)}],
                    }, ensure_ascii=False) + "\n")
                    added += 1
            print(f"[MIXED] QASPER 실제 인간 논문 QA {added}건 혼합 완료")
        except Exception as exc:
            print(f"[INFO] QASPER 혼합 생략 (외부 arXiv 역합성만 사용): {exc}")
else:
    print("RUN_QA_SYNTH=False")

In [ ]:
# Cell 5.2 — [Exp 3A] `Qwen/Qwen3-4B-Instruct-2507` Completion-Only LoRA SFT (6144 ctx) + 배치 DEV 평가
SFT_ADAPTER_DIR = MODELS_DIR / "qwen3_4b_2507_5chunk_sft_lora"
SFT_MAX_LENGTH = 6144

if RUN_SFT_TRAIN:
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer

    cleanup_all_gpu_llms()
    set_encoders_device("cpu")

    sft_rows = read_jsonl(SFT_DATA_PATH)
    print(f"SFT 학습 샘플 수: {len(sft_rows)} | 유형 분포: {Counter(r['q_type'] for r in sft_rows)}")

    tok = AutoTokenizer.from_pretrained(BASE_STUDENT_MODEL, padding_side="right")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def _to_prompt_completion(row):
        p_msgs = row["messages"][:-1]
        c_text = row["messages"][-1]["content"] + tok.eos_token
        return {"prompt": _render_prompt(tok, p_msgs, enable_thinking=False), "completion": c_text}

    ds = Dataset.from_list([_to_prompt_completion(r) for r in sft_rows]).train_test_split(test_size=0.05, seed=378)
    model = AutoModelForCausalLM.from_pretrained(BASE_STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    model.config.use_cache = False
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    lora_cfg = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    sft_sig = inspect.signature(SFTConfig.__init__).parameters
    cfg_kwargs = {
        "output_dir": str(SFT_ADAPTER_DIR / "checkpoints"),
        "num_train_epochs": 2, "per_device_train_batch_size": 1, "gradient_accumulation_steps": 16,
        "learning_rate": 8e-5, "lr_scheduler_type": "cosine", "warmup_steps": 15,
        "bf16": True, "gradient_checkpointing": True, "logging_steps": 10,
        "eval_strategy": "epoch", "save_strategy": "epoch", "save_total_limit": 1,
        "seed": 378, "report_to": "none",
    }
    if "gradient_checkpointing_kwargs" in sft_sig:
        cfg_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    if "max_length" in sft_sig:
        cfg_kwargs["max_length"] = SFT_MAX_LENGTH
    elif "max_seq_length" in sft_sig:
        cfg_kwargs["max_seq_length"] = SFT_MAX_LENGTH
    if "completion_only_loss" in sft_sig:
        cfg_kwargs["completion_only_loss"] = True
        print("[OK] completion_only_loss=True 활성 (어시스턴트 JSON 출력만 학습!)")

    t_sig = inspect.signature(SFTTrainer.__init__).parameters
    tk = {"model": model, "args": SFTConfig(**{k: v for k, v in cfg_kwargs.items() if k in sft_sig}), "train_dataset": ds["train"], "eval_dataset": ds["test"], "peft_config": lora_cfg}
    tk["processing_class" if "processing_class" in t_sig else "tokenizer"] = tok

    trainer = SFTTrainer(**tk)
    trainer.train()
    SFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(SFT_ADAPTER_DIR))
    tok.save_pretrained(str(SFT_ADAPTER_DIR))
    shutil.rmtree(SFT_ADAPTER_DIR / "checkpoints", ignore_errors=True)
    print(f"[SAVED] SFT LoRA 어댑터: {SFT_ADAPTER_DIR}")
    try:
        trainer.model.cpu()
        trainer.accelerator.free_memory()
    except Exception:
        pass
    try:
        model.cpu()
    except Exception:
        pass
    free_vram("model", "trainer", "ds")

    # 학습 직후 Exp 3A 배치 DEV 평가 (~4분)
    set_encoders_device(DEVICE)
    sft_llm, sft_tok = load_hf_llm(BASE_STUDENT_MODEL, load_in_4bit=False, lora_adapter=SFT_ADAPTER_DIR)
    active_nli = SlidingWindowScientificNLI(NLI_SAVE_DIR, device=DEVICE) if NLI_SAVE_DIR.exists() else None
    exp3a_records, exp3a_report = run_pipeline_eval(
        sft_llm, sft_tok, exp_name="exp3a_qwen3_4b_2507_comp_only_sft", split="dev",
        batch_size=BATCH_SIZE_4B, nli_scorer=active_nli, nli_contra_threshold=0.70, nli_post_entail_min=None,
        notes="Phase 3A: Reverse 5-Chunk Completion-Only LoRA SFT (6144 ctx) + Pre-NLI",
    )
    compare_experiments_statistically(exp1c_report, exp3a_report)
else:
    print("RUN_SFT_TRAIN=False")

In [ ]:
# Cell 5.3 — 병렬 On-Policy DPO 롤아웃 수집기 (4 prompts x 4 seqs = 16 병렬 생성, ~25분!)
DPO_DATA_PATH = DATASETS_DIR / "onpolicy_dpo_en_ko_v4.jsonl"

if RUN_ONPOLICY_DPO_COLLECT:
    if "sft_llm" not in globals() or not _vllm_is_alive(globals().get("sft_llm")):
        cleanup_all_gpu_llms()
        sft_llm, sft_tok = load_hf_llm(BASE_STUDENT_MODEL, load_in_4bit=False, lora_adapter=SFT_ADAPTER_DIR)

    sft_rows = [r for r in read_jsonl(SFT_DATA_PATH) if not r["paper_id"].startswith("qasper-")]
    done_keys = {(r["paper_id"], r["q_type"]) for r in read_jsonl(DPO_DATA_PATH)} if DPO_DATA_PATH.exists() else set()
    pending_rows = [r for r in sft_rows if (r["paper_id"], r["q_type"]) not in done_keys]
    print(f"[DPO ROLLOUT] 전체 {len(sft_rows)}건 중 남은 {len(pending_rows)}건 병렬 스캔 (4x4=16병렬)")

    t0 = time.perf_counter()
    with DPO_DATA_PATH.open("a", encoding="utf-8") as dpo_f:
        macro = DPO_ROLLOUT_BATCH * 4
        for start in range(0, len(pending_rows), macro):
            sub = pending_rows[start:start + macro]
            rollouts_list = generate_chat_batch(
                sft_llm, sft_tok, [r["messages"][:-1] for r in sub],
                max_new_tokens=896, batch_size=DPO_ROLLOUT_BATCH,
                temperature=0.8, num_return_sequences=4,
            )
            for row, cands in zip(sub, rollouts_list):
                chosen_content = row["messages"][-1]["content"]
                exp_v, lang, ev = row["expected_verdict"], row["language"], row["evidence"]
                atomic_facts = row.get("atomic_facts") or []
                rejected_content, fail_tag = None, None
                for cand in cands:
                    if not cand or len(cand.strip()) < 20 or cand.strip() == chosen_content.strip():
                        continue
                    chk = verify_and_assemble_v2(cand, ev, lang)
                    rv = chk["pipeline"].get("raw_verdict")
                    if exp_v == "contradicted" and rv == "supported":
                        rejected_content, fail_tag = cand, "sycophancy_false_premise"
                    elif exp_v == "supported" and rv in {"unsupported", "contradicted"}:
                        rejected_content, fail_tag = cand, "over_refusal"
                    elif rv != exp_v:
                        rejected_content, fail_tag = cand, "wrong_verdict"
                    elif not chk["pipeline"].get("quote_verified"):
                        rejected_content, fail_tag = cand, "bad_quote"
                    elif not chk["pipeline"].get("numeric_verified"):
                        rejected_content, fail_tag = cand, "bad_numeric"
                    elif chk["pipeline"].get("fallback_used"):
                        rejected_content, fail_tag = cand, "json_error"
                    elif row.get("q_type") == "ans_en" and atomic_facts:
                        ans_low = chk["response"]["answer"].casefold()
                        if sum(f.casefold() in ans_low for f in atomic_facts) < len(atomic_facts):
                            rejected_content, fail_tag = cand, "incomplete_facts"
                    if rejected_content:
                        break
                if rejected_content:
                    dpo_f.write(json.dumps({
                        "paper_id": row["paper_id"], "q_type": row["q_type"], "language": lang,
                        "fail_tag": fail_tag, "prompt": row["messages"][:-1],
                        "chosen": chosen_content, "rejected": rejected_content,
                    }, ensure_ascii=False) + "\n")
            dpo_f.flush()
            done_n = min(start + macro, len(pending_rows))
            print(f"On-Policy 병렬 롤아웃: {done_n}/{len(pending_rows)} ({(time.perf_counter()-t0)/60:.1f}분 경과)")

    collected = read_jsonl(DPO_DATA_PATH)
    print(f"[DONE] 수집된 On-Policy DPO 쌍: {len(collected)}건 | fail_tag={Counter(r.get('fail_tag') for r in collected)}")
else:
    print("RUN_ONPOLICY_DPO_COLLECT=False")

In [ ]:
# Cell 5.4 — [Exp 3B] SFT-정규화 On-Policy DPO (`max_prompt_length=4096, max_length=5120`) + 배치 DEV 평가
DPO_ADAPTER_DIR = MODELS_DIR / "qwen3_4b_2507_onpolicy_dpo_lora"

if RUN_DPO_TRAIN and DPO_DATA_PATH.exists() and len(read_jsonl(DPO_DATA_PATH)) >= 20:
    from datasets import Dataset
    from peft import LoraConfig, PeftModel
    from trl import DPOConfig, DPOTrainer

    cleanup_all_gpu_llms()
    set_encoders_device("cpu")

    dpo_rows = read_jsonl(DPO_DATA_PATH)
    print(f"On-Policy DPO 쌍 수: {len(dpo_rows)} | fail_tag={Counter(r.get('fail_tag') for r in dpo_rows)}")
    tok = AutoTokenizer.from_pretrained(BASE_STUDENT_MODEL, padding_side="right")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def _fmt(r):
        return {"prompt": _render_prompt(tok, r["prompt"], enable_thinking=False), "chosen": r["chosen"], "rejected": r["rejected"]}

    dpo_ds = Dataset.from_list([_fmt(r) for r in dpo_rows]).train_test_split(test_size=0.05, seed=378)
    # SFT LoRA를 베이스에 인메모리 병합한 뒤 새 DPO LoRA를 부착 -> ref_model=None이 정확히 pi_SFT와 일치 (추가 VRAM 0!)
    base = AutoModelForCausalLM.from_pretrained(BASE_STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
    model = PeftModel.from_pretrained(base, str(SFT_ADAPTER_DIR)).merge_and_unload() if SFT_ADAPTER_DIR.exists() else base
    model.config.use_cache = False
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    dpo_lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

    dpo_sig = inspect.signature(DPOConfig.__init__).parameters
    dpo_kwargs = {
        "output_dir": str(DPO_ADAPTER_DIR / "checkpoints"),
        "beta": 0.1, "learning_rate": 2e-5, "num_train_epochs": 2,
        "per_device_train_batch_size": 1, "gradient_accumulation_steps": 8,
        "max_prompt_length": 3584, "max_length": 4224,
        "bf16": True, "gradient_checkpointing": True,
        "logging_steps": 10, "save_strategy": "epoch", "save_total_limit": 1,
        "seed": 378, "report_to": "none",
    }
    if "gradient_checkpointing_kwargs" in dpo_sig:
        dpo_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    if "loss_weights" in dpo_sig:
        dpo_kwargs["loss_type"] = ["sigmoid", "sft"]
        dpo_kwargs["loss_weights"] = [1.0, 0.25]
        print("[OK] Modern TRL composite loss 활성: loss_type=['sigmoid', 'sft'], weights=[1.0, 0.25]")
    elif "rpo_alpha" in dpo_sig:
        dpo_kwargs["rpo_alpha"] = 0.25
        print("[OK] TRL rpo_alpha=0.25 활성")

    t_sig = inspect.signature(DPOTrainer.__init__).parameters
    filtered_dpo = {k: v for k, v in dpo_kwargs.items() if k in dpo_sig}
    try:
        dpo_cfg = DPOConfig(**filtered_dpo)
    except Exception:
        filtered_dpo.pop("loss_weights", None)
        filtered_dpo["loss_type"] = "sigmoid"
        dpo_cfg = DPOConfig(**filtered_dpo)
    tk = {
        "model": model, "ref_model": None, "args": dpo_cfg,
        "train_dataset": dpo_ds["train"], "eval_dataset": dpo_ds["test"], "peft_config": dpo_lora_cfg,
    }
    tk["processing_class" if "processing_class" in t_sig else "tokenizer"] = tok

    trainer = DPOTrainer(**tk)
    trainer.train()
    DPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(DPO_ADAPTER_DIR))
    tok.save_pretrained(str(DPO_ADAPTER_DIR))
    (DPO_ADAPTER_DIR / "adapter_chain.json").write_text(
        json.dumps({"base_model": BASE_STUDENT_MODEL, "adapters_in_order": [SFT_ADAPTER_DIR.name, DPO_ADAPTER_DIR.name]}, indent=2) + "\n",
        encoding="utf-8",
    )
    shutil.rmtree(DPO_ADAPTER_DIR / "checkpoints", ignore_errors=True)
    print(f"[SAVED] On-Policy DPO LoRA 어댑터 (stacked on SFT): {DPO_ADAPTER_DIR}")
    try:
        trainer.model.cpu()
        trainer.accelerator.free_memory()
    except Exception:
        pass
    for _m in (model, base):
        try:
            _m.cpu()
        except Exception:
            pass
    free_vram("model", "base", "trainer", "dpo_ds")

    # 학습 직후 Exp 3B 배치 DEV 평가 (~4분, SFT -> DPO 순차 병합 적용!)
    set_encoders_device(DEVICE)
    dpo_llm, dpo_tok = load_hf_llm(BASE_STUDENT_MODEL, load_in_4bit=False, lora_adapter=[SFT_ADAPTER_DIR, DPO_ADAPTER_DIR])
    active_nli = SlidingWindowScientificNLI(NLI_SAVE_DIR, device=DEVICE) if NLI_SAVE_DIR.exists() else None
    exp3b_records, exp3b_report = run_pipeline_eval(
        dpo_llm, dpo_tok, exp_name="exp3b_qwen3_4b_2507_onpolicy_rpo_dpo", split="dev",
        batch_size=BATCH_SIZE_4B, nli_scorer=active_nli, nli_contra_threshold=0.70, nli_post_entail_min=None,
        notes="Phase 3B: SFT-Regularized On-Policy DPO (exact pi_ref=pi_SFT) + Pre-NLI",
    )
    if "exp3a_report" in globals():
        compare_experiments_statistically(exp3a_report, exp3b_report)
else:
    print("RUN_DPO_TRAIN skipped")

## Section 6. [Phase 4] Conformal 보정 · Holdout 회귀 비교 · RTX 3060 내보내기 · 최종 일괄 반출

In [ ]:
# Cell 6.1 — Paper-Grouped 5-Fold OOF Conformal Gate + 최종 모델 `corpus30` Holdout(240문항) 배치 회귀 비교
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def fit_grouped_conformal_gate(dev_records: list[dict], alphas=(0.02, 0.05, 0.10)) -> dict:
    feats_list, y_list, groups = [], [], []
    feat_names = [
        "top_reranker_score", "score_margin_1_5", "is_ko",
        "nli_max_entailment", "nli_max_contradiction",
        "quote_verified_ratio", "numeric_verified_ratio",
    ]
    for rec in dev_records:
        q = QUESTION_MAP[rec["case_id"]]
        pipe = rec["output"]["pipeline"]
        feats_list.append([
            float(pipe.get("top_reranker_score", 0.0)),
            float(pipe.get("score_margin_1_5", 0.0)),
            1.0 if q["language"] == "ko" else 0.0,
            float(pipe.get("nli_max_entailment", 0.5)),
            float(pipe.get("nli_max_contradiction", 0.0)),
            float(pipe.get("quote_verified_ratio", 0.0)),
            float(pipe.get("numeric_verified_ratio", 0.0)),
        ])
        y_list.append(1 if q["answerability"] == "answerable" else 0)
        groups.append(q["paper_id"])

    X, y, g = np.asarray(feats_list, dtype=np.float64), np.asarray(y_list, dtype=np.int64), np.asarray(groups)
    gkf = GroupKFold(n_splits=min(5, len(set(groups))))
    oof_probs = np.zeros(len(y), dtype=np.float64)
    for tr_i, va_i in gkf.split(X, y, g):
        m = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(C=0.5, max_iter=1000))])
        m.fit(X[tr_i], y[tr_i])
        oof_probs[va_i] = m.predict_proba(X[va_i])[:, 1]

    neg_oof = np.sort(oof_probs[y == 0])
    n_neg = len(neg_oof)
    rc_table, thresholds = [], {}
    for a in alphas:
        q_idx = min(n_neg - 1, max(0, int(np.ceil((1.0 - a) * (n_neg + 1))) - 1))
        thr = float(neg_oof[q_idx])
        thresholds[f"{a:.2f}"] = thr
        pred_pos = oof_probs >= thr
        rc_table.append({
            "alpha": a, "oof_threshold": round(thr, 4),
            "answerable_coverage": round(float(pred_pos[y == 1].mean()), 3),
            "negative_leak_rate": round(float(pred_pos[y == 0].mean()), 3),
        })

    order = np.argsort(-oof_probs)
    aurc = float(np.mean(np.cumsum(1.0 - y[order]) / np.arange(1, len(y) + 1)))

    final_clf = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(C=0.5, max_iter=1000))])
    final_clf.fit(X, y)
    payload = {
        "schema_version": 2, "calibration_method": "paper_grouped_5fold_oof_split_conformal",
        "feature_names": feat_names,
        "scaler_mean": final_clf.named_steps["scaler"].mean_.tolist(),
        "scaler_scale": final_clf.named_steps["scaler"].scale_.tolist(),
        "coef": final_clf.named_steps["lr"].coef_[0].tolist(),
        "intercept": float(final_clf.named_steps["lr"].intercept_[0]),
        "conformal_thresholds_by_alpha": thresholds, "recommended_alpha": 0.05,
        "conformal_threshold": thresholds["0.05"], "oof_aurc": round(aurc, 4), "oof_risk_coverage": rc_table,
    }
    out_path = EXPORTS_DIR / "calibrated_gate.json"
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    sync_to_gcs("exports/calibrated_gate.json")
    print(f"[SAVED] {out_path} | OOF AURC={aurc:.4f}")
    display(pd.DataFrame(rc_table))
    return payload


best_dev_name = next((n for n in ("exp3b_records", "exp3a_records", "exp2b_records", "exp1c_records") if n in globals()), None)
if best_dev_name:
    print(f"Conformal Calibration using {best_dev_name}:")
    _ = fit_grouped_conformal_gate(globals()[best_dev_name])

if RUN_FINAL_HOLDOUT_REGRESSION:
    BEST_ADAPTERS = [d for d in (SFT_ADAPTER_DIR, DPO_ADAPTER_DIR) if d.exists()]
    cleanup_all_gpu_llms()
    set_encoders_device(DEVICE)
    final_llm, final_tok = load_hf_llm(BASE_STUDENT_MODEL, load_in_4bit=False, lora_adapter=BEST_ADAPTERS or None)
    final_nli = SlidingWindowScientificNLI(NLI_SAVE_DIR, device=DEVICE) if NLI_SAVE_DIR.exists() else None
    holdout_v2_records, holdout_v2_report = run_pipeline_eval(
        final_llm, final_tok,
        exp_name="exp4_v2_final_regression", split="holdout",
        batch_size=BATCH_SIZE_4B, nli_scorer=final_nli, nli_contra_threshold=0.70,
        nli_post_entail_min=None,
        notes=f"Final v2 regression on corpus30 holdout (adapters={[a.name for a in BEST_ADAPTERS]})",
    )
    print("\n=== v1 Final Holdout vs v2 Final Holdout 통계 유의성 비교 (240 outputs) ===")
    compare_experiments_statistically(V1_HOLDOUT_REPORT, holdout_v2_report)
    cleanup_all_gpu_llms()

    if RUN_SEMANTIC_JUDGE:
        llm_14b, tok_14b = load_hf_llm(TEACHER_MODEL_NAME, load_in_4bit=True)
        _ = run_semantic_judge(holdout_v2_records, llm_14b, tok_14b, "exp4_v2_final_holdout", batch_size=BATCH_SIZE_14B)
        cleanup_all_gpu_llms()

In [ ]:
# Cell 6.2 — 로컬 RTX 3060 6GB용 산출물 내보내기 (SFT+DPO 순차 병합 GGUF Q4_K_M + ONNX INT8, 임시 파일 자동 정리)
if RUN_EXPORT_GGUF:
    from peft import PeftModel
    cleanup_all_gpu_llms()
    set_encoders_device("cpu")
    cleanup_disk_caches(("Qwen3-14B-AWQ", "Qwen3-14B-FP8", "Qwen3-Embedding-0.6B", "Qwen3-Reranker-0.6B"))
    ADAPTERS_TO_MERGE = [d for d in (SFT_ADAPTER_DIR, DPO_ADAPTER_DIR) if d.exists()]
    MERGED_DIR = Path("/content/merged_qwen3_4b_2507")
    print(f"Merging adapters in order: {[a.name for a in ADAPTERS_TO_MERGE]} -> {MERGED_DIR}")
    merged = AutoModelForCausalLM.from_pretrained(BASE_STUDENT_MODEL, torch_dtype=torch.float16, device_map="cpu")
    tok = AutoTokenizer.from_pretrained(BASE_STUDENT_MODEL)
    for ad in ADAPTERS_TO_MERGE:
        merged = PeftModel.from_pretrained(merged, str(ad)).merge_and_unload()
    merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    tok.save_pretrained(str(MERGED_DIR))
    del merged
    gc.collect()

    try:
        if not Path("/content/llama.cpp").exists():
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/llama.cpp/gguf-py"], check=False)
        qbin = Path("/content/llama.cpp/build/bin/llama-quantize")
        if not qbin.exists():
            subprocess.run(["cmake", "-B", "/content/llama.cpp/build", "/content/llama.cpp"], check=True)
            subprocess.run(["cmake", "--build", "/content/llama.cpp/build", "--target", "llama-quantize", "-j4"], check=True)
        subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", str(MERGED_DIR), "--outfile", "/content/qwen3-4b-2507-f16.gguf", "--outtype", "f16"], check=True)
        GGUF_OUT = EXPORTS_DIR / "qwen3-4b-2507-paper-qna-q4_k_m.gguf"
        subprocess.run([str(qbin), "/content/qwen3-4b-2507-f16.gguf", str(GGUF_OUT), "Q4_K_M"], check=True)
        print(f"[EXPORTED] {GGUF_OUT} ({GGUF_OUT.stat().st_size / (1024**3):.2f} GiB)")
        print("로컬 RTX 3060 6GB 실행 명령:")
        print(f"  llama-server.exe -m outputs/exports/{GGUF_OUT.name} -ngl 99 -fa -ctk q8_0 -ctv q8_0 -c 6144 --port 8080")
        sync_to_gcs(f"exports/{GGUF_OUT.name}")
    except Exception as exc:
        print(f"[WARN] GGUF 내보내기 오류 (어댑터는 {MODELS_DIR}에 안전 보존됨): {exc}")
    finally:
        shutil.rmtree(MERGED_DIR, ignore_errors=True)
        Path("/content/qwen3-4b-2507-f16.gguf").unlink(missing_ok=True)

if RUN_EXPORT_NLI_ONNX and NLI_SAVE_DIR.exists():
    ONNX_OUT = EXPORTS_DIR / "scientific_nli_int8"
    ONNX_OUT.mkdir(parents=True, exist_ok=True)
    exported_ok = False
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "optimum[onnxruntime]"], check=False)
        r1 = subprocess.run(["optimum-cli", "export", "onnx", "--model", str(NLI_SAVE_DIR), "--task", "text-classification", "/content/nli_onnx_fp32"], check=False)
        if r1.returncode == 0:
            r2 = subprocess.run(["optimum-cli", "onnxruntime", "quantize", "--onnx_model", "/content/nli_onnx_fp32", "--avx2", "-o", str(ONNX_OUT)], check=False)
            exported_ok = (r2.returncode == 0 and any(ONNX_OUT.glob("*.onnx")))
    except Exception as exc:
        print(f"[INFO] optimum-cli 실패 -> PyTorch ONNX 폴백 시도 ({exc})")

    if not exported_ok:
        try:
            from onnxruntime.quantization import QuantType, quantize_dynamic
            from transformers import AutoModelForSequenceClassification, AutoTokenizer
            ntok = AutoTokenizer.from_pretrained(str(NLI_SAVE_DIR))
            nmod = AutoModelForSequenceClassification.from_pretrained(str(NLI_SAVE_DIR)).cpu().eval()
            dummy = ntok("Paper premise text.", "Question hypothesis.", return_tensors="pt")
            fp32_onnx = Path("/content/nli_model_fp32.onnx")
            torch.onnx.export(
                nmod, (dummy["input_ids"], dummy["attention_mask"]), str(fp32_onnx),
                input_names=["input_ids", "attention_mask"], output_names=["logits"],
                dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}, "logits": {0: "batch"}},
                opset_version=17,
            )
            quantize_dynamic(str(fp32_onnx), str(ONNX_OUT / "model_quantized.onnx"), weight_type=QuantType.QInt8)
            ntok.save_pretrained(str(ONNX_OUT))
            nmod.config.save_pretrained(str(ONNX_OUT))
            fp32_onnx.unlink(missing_ok=True)
            exported_ok = (ONNX_OUT / "model_quantized.onnx").exists()
        except Exception as exc:
            print(f"[WARN] ONNX INT8 폴백도 건너뜀 (원본 HF 모델 {NLI_SAVE_DIR} 사용 가능): {exc}")

    shutil.rmtree("/content/nli_onnx_fp32", ignore_errors=True)
    if exported_ok:
        print(f"[EXPORTED] Scientific NLI ONNX INT8 -> {ONNX_OUT}")
        sync_to_gcs("exports/scientific_nli_int8")

In [ ]:
# Cell 6.3 — 신규 미공개 평가셋 (`corpus-v2-holdout`, 30편, HF Parquet validation 스플릿 0% 겹침 동결, ~3초!)
if RUN_BUILD_HOLDOUT_V2:
    from datasets import load_dataset

    h2_chunks_path = HOLDOUT_V2_DIR / "chunks.jsonl"
    h2_meta_path = HOLDOUT_V2_DIR / "manifest_v2.json"
    train_ids = {m["arxiv_id"] for m in read_jsonl(EXT_META_PATH)} if EXT_META_PATH.exists() else set()
    papers_v2, chunks_v2 = [], []

    try:
        val_iter = load_dataset("ccdv/arxiv-summarization", "document", split="validation", streaming=True)
    except Exception:
        val_iter = pd.read_parquet("https://huggingface.co/datasets/ccdv/arxiv-summarization/resolve/main/document/validation-00000-of-00002.parquet").to_dict(orient="records")

    for v_i, ex in enumerate(val_iter):
        if len(papers_v2) >= HOLDOUT_V2_PAPERS:
            break
        ab = repair_pdf_chunk_text(str(ex.get("abstract") or ""))
        art = repair_pdf_chunk_text(str(ex.get("article") or ""))
        if len(ab) < 200 or len(art) < 6000:
            continue
        head_norm = ab[:200].casefold()
        if any(ft in head_norm for ft in FORBIDDEN_TITLES if len(ft) > 20):
            continue
        b_chs = _simple_chunk_text(art, chunk_size=1200, overlap=200)[:23]
        if len(b_chs) < 10:
            continue
        pid = f"v2-paper-{len(papers_v2)+1:03d}"
        p_chunks = [{"paper_id": pid, "chunk_id": f"{pid}-p001-c001", "page": 1, "section": "Abstract", "text": ab[:1400]}]
        for ci, txt in enumerate(b_chs, start=1):
            pno = min(15, 1 + (ci // 2))
            p_chunks.append({"paper_id": pid, "chunk_id": f"{pid}-p{pno:03d}-c{ci+1:03d}", "page": pno, "section": "Body", "text": txt})
        papers_v2.append({"paper_id": pid, "arxiv_id": f"hf-ccdv-val-{v_i}", "title": ab[:120], "page_count": max(c["page"] for c in p_chunks), "chunk_count": len(p_chunks)})
        chunks_v2.extend(p_chunks)

    h2_chunks_path.write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in chunks_v2) + "\n", encoding="utf-8")
    h2_meta_path.write_text(json.dumps({"schema_version": 2, "source": "ccdv/arxiv-summarization:validation", "papers": papers_v2}, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    freeze = {
        "schema_version": 2, "created_at": datetime.now(timezone.utc).isoformat(),
        "excluded_corpus30_count": len(FORBIDDEN_ARXIV_IDS), "excluded_train_count": len(train_ids),
        "sha256": {
            "chunks.jsonl": hashlib.sha256(h2_chunks_path.read_bytes()).hexdigest(),
            "manifest_v2.json": hashlib.sha256(h2_meta_path.read_bytes()).hexdigest(),
        },
    }
    (HOLDOUT_V2_DIR / "freeze_v2.json").write_text(json.dumps(freeze, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"[FROZEN] corpus-v2-holdout {len(papers_v2)}편 ({len(chunks_v2)}청크, validation 스플릿) 동결 완료", flush=True)

In [ ]:
# Cell 6.4 — [원샷 완료!] 전체 산출물 요약 & Google Drive / GCS / 로컬 일괄 반출
COPY_FINAL_TO_GDRIVE = False  # 원샷 완료 후 Google Drive로 한 번에 복사할 때 True로 바꾸고 이 셀만 실행!

print("=== 최종 누적 실험 비교표 ===")
show_experiment_table()

print("\n=== 생성된 최종 산출물 목록 ===")
for sub in ("exports", "runs", "corpus_v2_holdout"):
    p = WORK_DIR / sub
    if p.exists():
        total_mb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024 ** 2)
        print(f"  {sub:20s}: {total_mb:7.1f} MiB ({sum(1 for f in p.rglob('*') if f.is_file())} files)")

REPORT_ZIP = WORK_DIR / "paper_qna_v2_reports_and_gate.zip"
with zipfile.ZipFile(REPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in (RUNS_DIR, HOLDOUT_V2_DIR):
        for f in folder.rglob("*"):
            if f.is_file() and f.suffix != ".npz":
                zf.write(f, arcname=f.relative_to(WORK_DIR))
    gate_f = EXPORTS_DIR / "calibrated_gate.json"
    if gate_f.exists():
        zf.write(gate_f, arcname="exports/calibrated_gate.json")
print(f"\n[READY] 리포트·로그·보정기 번들: {REPORT_ZIP} ({REPORT_ZIP.stat().st_size / (1024**2):.2f} MiB)")
print(f"[READY] 로컬 RTX 3060용 GGUF / ONNX 폴더: {EXPORTS_DIR}")

if COPY_FINAL_TO_GDRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dst_drive = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME / "final_results"
    dst_drive.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPORT_ZIP, dst_drive / REPORT_ZIP.name)
    if EXPORTS_DIR.exists():
        shutil.copytree(EXPORTS_DIR, dst_drive / "exports", dirs_exist_ok=True)
    if MODELS_DIR.exists():
        shutil.copytree(MODELS_DIR, dst_drive / "models", dirs_exist_ok=True)
    print(f"[COPIED TO DRIVE] {dst_drive}")

## Section 7. 추가 튜닝 샌드박스 (원샷 완료 후 셀을 붙여가며 추가 개선)

- 원샷 완주 후 비교표에서 아쉬운 지표가 있으면 아래에 셀을 붙여 해당 구간만 빠르게 재튜닝하고 `log_experiment()`로 기록하세요.

In [ ]:
# Cell 7.1 — 누적 실험 비교표 다시 보기
show_experiment_table()